# AnimeLoom — Character Training & Animation Pipeline (Google Colab)

Train character LoRAs for consistent anime identity across long-form animation.
Supports **SDXL** (Animagine XL 3.1) for keyframes and **SD 1.5** (DreamShaper 8) for AnimateDiff video.

**What this notebook does:**
1. Sets up AnimeLoom environment on Colab
2. Mounts Google Drive for persistent storage
3. Downloads datasets or uploads your own character images
4. Auto-captions images with BLIP
5. Trains character LoRAs (SDXL + SD 1.5 for AnimateDiff)
6. Tests trained characters for consistency
7. Exports everything for AnimeLoom's pipeline
8. **Generates animated video** — single clips or 2+ minute long videos
9. **NEW: Text-to-Anime** — type a story, get an anime video (Sora-like)

| Colab Tier | GPU | Session Limit | Good For |
|------------|-----|---------------|----------|
| Free | T4 (15GB) | ~90 min | 1–2 characters, short clips |
| Pro ($10) | T4/A100 | ~12 hours | Full training + 2-min video generation + text-to-anime |

---

## 1. Environment Setup
Clones AnimeLoom, installs dependencies, mounts Google Drive.

In [ ]:
#@title 1. Setup AnimeLoom Environment
#@markdown Clones the repo, installs deps, and mounts Google Drive.

import os, sys, subprocess
from pathlib import Path

# --- Clone AnimeLoom ---
REPO_URL = "https://github.com/JoelJohnsonThomas/AnimeLoom.git"  #@param {type:"string"}
REPO_DIR = Path("/content/AnimeLoom")

if not REPO_DIR.exists():
    print("Cloning AnimeLoom…")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"AnimeLoom already at {REPO_DIR}")

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

# --- Mount Google Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Warehouse on Drive (persists across sessions) ---
WAREHOUSE = Path("/content/drive/MyDrive/AniLoom/warehouse")
WAREHOUSE.mkdir(parents=True, exist_ok=True)
os.environ["AI_CACHE_ROOT"] = str(WAREHOUSE)

for sub in ["lora", "models", "datasets/raw", "datasets/tagged", "outputs", "checkpoints"]:
    (WAREHOUSE / sub).mkdir(parents=True, exist_ok=True)

print(f"Warehouse: {WAREHOUSE}")

# --- Install dependencies ---
print("\nInstalling dependencies…")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cu118"],
    capture_output=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "diffusers>=0.24.0", "transformers>=4.35.0", "accelerate", "safetensors",
     "peft", "bitsandbytes", "xformers",
     "opencv-python-headless", "pillow", "numpy", "scikit-image",
     "datasets", "huggingface-hub", "tqdm",
     "controlnet-aux>=0.0.7",
     "optimum-quanto",
     "gfpgan", "basicsr", "facexlib", "realesrgan",
     "google-genai", "google-generativeai",
     "gradio", "ultralytics"],
    capture_output=True,
)


# --- Fix torchvision compatibility for GFPGAN/basicsr ---
import torchvision
if not hasattr(torchvision.transforms, 'functional_tensor'):
    import torchvision.transforms.functional as _F
    import types as _types
    _ftmod = _types.ModuleType('torchvision.transforms.functional_tensor')
    for _attr in ['rgb_to_grayscale', 'normalize', 'resize', 'pad']:
        if hasattr(_F, _attr):
            setattr(_ftmod, _attr, getattr(_F, _attr))
    sys.modules['torchvision.transforms.functional_tensor'] = _ftmod
    torchvision.transforms.functional_tensor = _ftmod
    print("torchvision compatibility shim applied")

# --- Install RIFE for temporal upscaling ---
RIFE_DIR = WAREHOUSE / "models" / "Practical-RIFE"
if not RIFE_DIR.exists():
    print("Installing RIFE for frame interpolation...")
    subprocess.run(["git", "clone", "https://github.com/hzwer/Practical-RIFE", str(RIFE_DIR)], capture_output=True)
    import urllib.request
    train_log = RIFE_DIR / "train_log"
    train_log.mkdir(parents=True, exist_ok=True)
    flownet_path = train_log / "flownet.pkl"
    if not flownet_path.exists():
        try:
            urllib.request.urlretrieve(
                "https://github.com/hzwer/Practical-RIFE/releases/download/v4.6/flownet.pkl",
                str(flownet_path)
            )
            print("RIFE model weights downloaded")
        except Exception as e:
            print(f"RIFE download failed (temporal upscale will use fallback): {e}")
else:
    print("RIFE already installed")

import torch
print(f"\nPyTorch {torch.__version__}  CUDA {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\n✅ Setup complete!")

## 2. Get Character Images
Choose **one** of the three options below.

In [ ]:
#@title 2a. Download from HuggingFace (deepghs/character_similarity)
#@markdown Downloads pre-organised anime characters with multiple images each.

SPLIT = "v0_tiny"  #@param ["v0_tiny", "v1_pruned", "v1"]
MAX_CHARACTERS = 10  #@param {type:"slider", min:5, max:200, step:5}

from scripts.prepare_dataset import download_huggingface
download_huggingface(split=SPLIT, max_characters=MAX_CHARACTERS)

print(f"\nCharacter folders in {WAREHOUSE / 'datasets/raw'}:")
for d in sorted((WAREHOUSE / 'datasets/raw').iterdir()):
    if d.is_dir():
        n = len([f for f in d.iterdir() if f.suffix.lower() in {'.png','.jpg','.jpeg','.webp'}])
        print(f"  {d.name}: {n} images")

In [ ]:
#@title 2b. Upload your own images
#@markdown Upload images to a named character folder.

CHARACTER_NAME = "yuki"  #@param {type:"string"}

from google.colab import files
from pathlib import Path
import shutil, os

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
char_dir = WAREHOUSE / "datasets" / "raw" / CHARACTER_NAME.lower().replace(" ", "_")
char_dir.mkdir(parents=True, exist_ok=True)

print(f"Select 5–20 images of '{CHARACTER_NAME}' (different poses/expressions):")
uploaded = files.upload()

for fname, data in uploaded.items():
    dst = char_dir / fname
    dst.write_bytes(data)
    print(f"  Saved {fname} → {dst}")

total = len(list(char_dir.glob("*")))
print(f"\n✅ {total} images in {char_dir}")
if total < 5:
    print("⚠️  Recommend at least 5 images for good LoRA quality.")

In [ ]:
#@title 2c. Import from a Google Drive folder
#@markdown Point to a Drive folder containing character images.

CHARACTER_NAME = "yuki"  #@param {type:"string"}
DRIVE_FOLDER = "/content/drive/MyDrive/my_characters/yuki"  #@param {type:"string"}

from scripts.prepare_dataset import import_local
import_local(DRIVE_FOLDER, character_name=CHARACTER_NAME)

## 3. Auto-Caption Images with BLIP
Generates text captions for each image. Captions help the LoRA learn *what* it's seeing.

In [ ]:
#@title 3. Auto-Caption with BLIP
#@markdown Select which characters to caption (comma-separated), or "all".

CHARACTERS = "all"  #@param {type:"string"}

import os
from pathlib import Path
from scripts.prepare_dataset import caption_folder

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
raw_dir = WAREHOUSE / "datasets" / "raw"

if CHARACTERS.strip().lower() == "all":
    char_list = [d for d in sorted(raw_dir.iterdir()) if d.is_dir()]
else:
    char_list = [raw_dir / c.strip() for c in CHARACTERS.split(",")]

print(f"Captioning {len(char_list)} character(s)…\n")
for char_dir in char_list:
    if not char_dir.exists():
        print(f"⚠️  Skipping {char_dir.name} — folder not found")
        continue
    caption_folder(str(char_dir))
    print()

print("✅ Captioning complete!")

## 4. Train Character LoRA
This is the main training cell. Uses AnimeLoom's `scripts/train_lora.py` under the hood.

In [ ]:
#@title 4. Train a Single Character LoRA
#@markdown Configure training parameters and run.

CHARACTER_NAME = "yuki"  #@param {type:"string"}
USE_CAPTIONS = True  #@param {type:"boolean"}
LORA_RANK = 32  #@param {type:"slider", min:8, max:64, step:8}
LEARNING_RATE = 1e-4  #@param {type:"number"}
TRAIN_STEPS = 1000  #@param {type:"slider", min:200, max:3000, step:100}
BATCH_SIZE = 1  #@param {type:"slider", min:1, max:4, step:1}
RESOLUTION = 1024  #@param {type:"slider", min:256, max:1024, step:128}
BASE_MODEL = "cagliostrolab/animagine-xl-3.1"  #@param ["cagliostrolab/animagine-xl-3.1", "stabilityai/stable-diffusion-xl-base-1.0", "sd2-community/stable-diffusion-2-1"]

import os, torch
from pathlib import Path
from scripts.train_lora import train

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
char_id = CHARACTER_NAME.lower().replace(" ", "_")

if USE_CAPTIONS:
    image_dir = WAREHOUSE / "datasets" / "tagged" / char_id
else:
    image_dir = WAREHOUSE / "datasets" / "raw" / char_id

if not image_dir.exists():
    raise FileNotFoundError(
        f"No images at {image_dir}. "
        f"Run step 2 to get images, then step 3 to caption them."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "⚠️ No GPU!")
print(f"Training '{CHARACTER_NAME}' from {image_dir}")
print(f"Rank={LORA_RANK}  LR={LEARNING_RATE}  Steps={TRAIN_STEPS}  Res={RESOLUTION}")
print("=" * 60)

lora_dir = train(
    character_name=CHARACTER_NAME,
    image_dir=image_dir,
    rank=LORA_RANK,
    lr=LEARNING_RATE,
    steps=TRAIN_STEPS,
    batch_size=BATCH_SIZE,
    resolution=RESOLUTION,
    base_model=BASE_MODEL,
    use_captions=USE_CAPTIONS,
)

print(f"\n✅ LoRA saved to: {lora_dir}")
print("This persists on Google Drive — available in future sessions.")

In [ ]:
#@title 4b. Batch Train Multiple Characters
#@markdown Train several characters sequentially.

CHARACTER_NAMES = "yuki, kenji, sakura"  #@param {type:"string"}
USE_CAPTIONS = True  #@param {type:"boolean"}
LORA_RANK = 32  #@param {type:"slider", min:8, max:64, step:8}
LEARNING_RATE = 1e-4  #@param {type:"number"}
TRAIN_STEPS = 800  #@param {type:"slider", min:200, max:3000, step:100}
BATCH_SIZE = 1  #@param {type:"slider", min:1, max:4, step:1}
RESOLUTION = 1024  #@param {type:"slider", min:256, max:1024, step:128}
BASE_MODEL = "cagliostrolab/animagine-xl-3.1"  #@param ["cagliostrolab/animagine-xl-3.1", "stabilityai/stable-diffusion-xl-base-1.0", "sd2-community/stable-diffusion-2-1"]

import os, gc, torch
from pathlib import Path
from scripts.train_lora import train

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
char_list = [c.strip() for c in CHARACTER_NAMES.split(",")]

results = {}
for i, name in enumerate(char_list, 1):
    char_id = name.lower().replace(" ", "_")
    subdir = "tagged" if USE_CAPTIONS else "raw"
    image_dir = WAREHOUSE / "datasets" / subdir / char_id

    if not image_dir.exists():
        print(f"\n⚠️  [{i}/{len(char_list)}] Skipping '{name}' — no images at {image_dir}")
        results[name] = "SKIPPED"
        continue

    print(f"\n{'='*60}")
    print(f"[{i}/{len(char_list)}] Training '{name}'…")
    print(f"{'='*60}")

    try:
        lora_dir = train(
            character_name=name,
            image_dir=image_dir,
            rank=LORA_RANK,
            lr=LEARNING_RATE,
            steps=TRAIN_STEPS,
            batch_size=BATCH_SIZE,
            resolution=RESOLUTION,
            base_model=BASE_MODEL,
            use_captions=USE_CAPTIONS,
        )
        results[name] = f"OK → {lora_dir}"
    except Exception as e:
        print(f"❌ Failed: {e}")
        results[name] = f"FAILED: {e}"

    # Free VRAM between characters
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("Batch training results:")
for name, status in results.items():
    print(f"  {name}: {status}")

## 4c. Train SD 1.5 LoRA for AnimateDiff
Your SDXL LoRAs won't work with AnimateDiff (SD 1.5). Train a separate SD 1.5 LoRA using **DreamShaper 8** as the base model. This is required for animated video generation.

In [ ]:
#@title 4c. Train SD 1.5 LoRA for AnimateDiff
#@markdown Trains a separate SD 1.5 LoRA using DreamShaper 8 (anime SD 1.5 model).
#@markdown Required for AnimateDiff video generation. Saves to `warehouse/lora/{name}_sd15/`.

CHARACTER_NAMES = "yuki_nagato, denji, sakura_haruno"  #@param {type:"string"}
USE_CAPTIONS = True  #@param {type:"boolean"}
LORA_RANK = 32  #@param {type:"slider", min:8, max:64, step:8}
LEARNING_RATE = 1e-4  #@param {type:"number"}
TRAIN_STEPS = 800  #@param {type:"slider", min:200, max:3000, step:100}
BATCH_SIZE = 1  #@param {type:"slider", min:1, max:4, step:1}
BASE_MODEL = "Lykon/dreamshaper-8"  #@param ["Lykon/dreamshaper-8", "Lykon/dreamshaper-7", "runwayml/stable-diffusion-v1-5"]

import os, gc, torch
from pathlib import Path
from scripts.train_lora import train

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
char_list = [c.strip() for c in CHARACTER_NAMES.split(",")]

print(f"Training SD 1.5 LoRAs for AnimateDiff compatibility")
print(f"Base model: {BASE_MODEL}")
print(f"Resolution: 512 (auto-set for SD 1.5)")
print(f"Characters: {char_list}")
print("=" * 60)

results = {}
for i, name in enumerate(char_list, 1):
    char_id = name.lower().replace(" ", "_")
    subdir = "tagged" if USE_CAPTIONS else "raw"
    image_dir = WAREHOUSE / "datasets" / subdir / char_id

    if not image_dir.exists():
        print(f"\n⚠️  [{i}/{len(char_list)}] Skipping '{name}' — no images at {image_dir}")
        results[name] = "SKIPPED"
        continue

    # Check if SD 1.5 LoRA already exists
    sd15_dir = WAREHOUSE / "lora" / f"{char_id}_sd15"
    if (sd15_dir / "adapter_model.safetensors").exists() or (sd15_dir / "pytorch_lora_weights.safetensors").exists():
        print(f"\n[{i}/{len(char_list)}] '{name}' already has SD 1.5 LoRA at {sd15_dir} — skipping")
        results[name] = f"EXISTS → {sd15_dir}"
        continue

    print(f"\n{'='*60}")
    print(f"[{i}/{len(char_list)}] Training SD 1.5 LoRA for '{name}'…")
    print(f"{'='*60}")

    try:
        lora_dir = train(
            character_name=name,
            image_dir=image_dir,
            rank=LORA_RANK,
            lr=LEARNING_RATE,
            steps=TRAIN_STEPS,
            batch_size=BATCH_SIZE,
            resolution=512,
            base_model=BASE_MODEL,
            use_captions=USE_CAPTIONS,
        )
        results[name] = f"OK → {lora_dir}"

        # Register SD 1.5 LoRA in memory bank
        from director.memory_bank import AssetMemoryBank
        memory = AssetMemoryBank(str(WAREHOUSE))
        lora_path = lora_dir / "pytorch_lora_weights.safetensors"
        if not lora_path.exists():
            lora_path = lora_dir / "adapter_model.safetensors"
        memory.update_character_lora(name, str(lora_path), model_type="sd15")
        print(f"  Registered SD 1.5 LoRA in memory bank")

    except Exception as e:
        print(f"❌ Failed: {e}")
        results[name] = f"FAILED: {e}"

    gc.collect()
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("SD 1.5 LoRA training results:")
for name, status in results.items():
    print(f"  {name}: {status}")
print(f"\nSD 1.5 LoRAs saved to: warehouse/lora/{{name}}_sd15/")

## 5. Test Trained Character
Generate sample images with the trained LoRA to verify consistency.

In [ ]:
#@title 5. Test Character LoRA
#@markdown Generate images to verify the trained character looks consistent.

CHARACTER_NAME = "yuki_nagato"  #@param {type:"string"}
TEST_PROMPT = "yuki_nagato, 1girl, smiling, looking at viewer, anime portrait, detailed, masterpiece"  #@param {type:"string"}
NEGATIVE_PROMPT = "blurry, low quality, deformed, extra fingers, worst quality"  #@param {type:"string"}
NUM_IMAGES = 4  #@param {type:"slider", min:1, max:8, step:1}
GUIDANCE_SCALE = 7.5  #@param {type:"slider", min:1.0, max:15.0, step:0.5}
INFERENCE_STEPS = 30  #@param {type:"slider", min:10, max:50, step:5}
IMAGE_SIZE = 768  #@param {type:"slider", min:512, max:1024, step:128}

import os, json, gc, torch
from pathlib import Path
from diffusers import DPMSolverMultistepScheduler
import matplotlib.pyplot as plt

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
char_id = CHARACTER_NAME.lower().replace(" ", "_")
lora_dir = WAREHOUSE / "lora" / char_id

# Load metadata
meta_path = lora_dir / "metadata.json"
if not meta_path.exists():
    raise FileNotFoundError(f"No trained LoRA for '{CHARACTER_NAME}' at {lora_dir}")

meta = json.loads(meta_path.read_text())
is_sdxl = meta.get("is_sdxl", "xl" in meta.get("base_model", "").lower() or "animagine" in meta.get("base_model", "").lower())

print(f"Character: {meta['character_name']}")
print(f"Base model: {meta['base_model']} ({'SDXL' if is_sdxl else 'SD'})")
print(f"Rank: {meta['rank']}  Steps: {meta['training_steps']}  Images: {meta['num_images']}")

# Free VRAM from any prior work
gc.collect()
torch.cuda.empty_cache()

# Load pipeline
print("\nLoading model…")
if is_sdxl:
    from diffusers import StableDiffusionXLPipeline
    pipe = StableDiffusionXLPipeline.from_pretrained(
        meta["base_model"],
        torch_dtype=torch.float16,
    )
else:
    from diffusers import StableDiffusionPipeline
    pipe = StableDiffusionPipeline.from_pretrained(
        meta["base_model"],
        torch_dtype=torch.float16,
        safety_checker=None,
    )

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.to("cuda")

# Save VRAM on T4 — decode large images in slices/tiles
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()

# Load LoRA via PEFT (matches how train_lora.py saves)
from peft import PeftModel
pipe.unet = PeftModel.from_pretrained(pipe.unet, str(lora_dir))
pipe.unet.eval()
print("LoRA loaded via PEFT.")

# Generate
print(f"\nGenerating {NUM_IMAGES} images at {IMAGE_SIZE}x{IMAGE_SIZE}…")
images = []
for i in range(NUM_IMAGES):
    with torch.autocast("cuda"):
        result = pipe(
            TEST_PROMPT,
            negative_prompt=NEGATIVE_PROMPT,
            height=IMAGE_SIZE,
            width=IMAGE_SIZE,
            num_inference_steps=INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            generator=torch.Generator("cuda").manual_seed(42 + i),
        )
    images.append(result.images[0])

# Display
cols = min(NUM_IMAGES, 4)
rows = (NUM_IMAGES + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
if NUM_IMAGES == 1:
    axes = [axes]
else:
    axes = axes.flat if hasattr(axes, 'flat') else [axes]

for i, (ax, img) in enumerate(zip(axes, images)):
    ax.imshow(img)
    ax.set_title(f"Seed {42+i}")
    ax.axis("off")

# Hide empty subplots
for j in range(i + 1, len(list(axes)) if hasattr(axes, '__len__') else 0):
    axes[j].axis("off")

plt.suptitle(f"'{CHARACTER_NAME}' — consistency test", fontsize=14)
plt.tight_layout()
plt.show()

# Save test outputs
test_dir = WAREHOUSE / "outputs" / f"test_{char_id}"
test_dir.mkdir(parents=True, exist_ok=True)
for i, img in enumerate(images):
    img.save(test_dir / f"test_{i+1}.png")
print(f"\nSaved to {test_dir}")

## 6. Register Characters in AssetMemoryBank
Makes trained characters available to AnimeLoom's Director Agent.

In [ ]:
#@title 6. Register Characters in Memory Bank
#@markdown Registers all tagged characters and links their trained LoRAs.

import os, json
from pathlib import Path
from director.memory_bank import AssetMemoryBank

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
memory = AssetMemoryBank(str(WAREHOUSE))

tagged_dir = WAREHOUSE / "datasets" / "tagged"
lora_root = WAREHOUSE / "lora"

if not tagged_dir.exists():
    print("No tagged datasets. Run step 3 first.")
else:
    for char_dir in sorted(tagged_dir.iterdir()):
        if not char_dir.is_dir():
            continue

        name = char_dir.name
        images = sorted(
            str(f) for f in char_dir.iterdir()
            if f.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}
        )
        if not images:
            continue

        # Read description from first caption
        desc = ""
        txt = next(char_dir.glob("*.txt"), None)
        if txt:
            desc = txt.read_text().strip()

        # Create or skip
        existing = memory.get_character(name)
        if existing:
            print(f"  '{name}' already registered")
        else:
            cid = memory.create_character(name=name, images=images, description=desc)
            print(f"  Registered '{name}' ({len(images)} images) -> {cid}")

        # Link LoRA if trained (PEFT saves as adapter_model.safetensors)
        lora_dir = lora_root / name
        meta_path = lora_dir / "metadata.json"
        if meta_path.exists():
            # Find the actual weight file
            adapter_path = lora_dir / "adapter_model.safetensors"
            legacy_path = lora_dir / "pytorch_lora_weights.safetensors"
            if adapter_path.exists():
                lora_path = str(adapter_path)
            elif legacy_path.exists():
                lora_path = str(legacy_path)
            else:
                lora_path = str(adapter_path)  # store expected path
            memory.update_character_lora(name, lora_path)
            print(f"    Linked LoRA: {lora_path}")

    memory.save_checkpoint()
    print(f"\n✅ Memory bank saved. {len(memory.list_characters())} characters registered.")

## 7. Dashboard & Export

In [ ]:
#@title 7a. Training Dashboard
#@markdown View all trained characters and their stats.

import os, json
from pathlib import Path
import matplotlib.pyplot as plt

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
lora_root = WAREHOUSE / "lora"

if not lora_root.exists():
    print("No trained LoRAs yet.")
else:
    characters = [d for d in sorted(lora_root.iterdir()) if d.is_dir()]
    if not characters:
        print("No trained LoRAs yet.")
    else:
        names, n_images, sizes_mb = [], [], []
        print(f"{'Character':<20} {'Images':>7} {'Steps':>7} {'Rank':>5} {'Size':>8} {'Model'}")
        print("-" * 80)

        for char_dir in characters:
            meta_path = char_dir / "metadata.json"
            if not meta_path.exists():
                continue
            m = json.loads(meta_path.read_text())

            # PEFT saves as adapter_model.safetensors, fallback to pytorch_lora_weights.safetensors
            weight_file = char_dir / "adapter_model.safetensors"
            if not weight_file.exists():
                weight_file = char_dir / "pytorch_lora_weights.safetensors"
            size = weight_file.stat().st_size / 1e6 if weight_file.exists() else 0

            names.append(m.get("character_name", char_dir.name))
            n_images.append(m.get("num_images", 0))
            sizes_mb.append(round(size, 1))

            print(
                f"{m.get('character_name', '?'):<20} "
                f"{m.get('num_images', '?'):>7} "
                f"{m.get('training_steps', '?'):>7} "
                f"{m.get('rank', '?'):>5} "
                f"{size:>7.1f}M "
                f"{m.get('base_model', '?').split('/')[-1]}"
            )

        if names:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
            ax1.barh(names, n_images, color="#4e79a7")
            ax1.set_xlabel("Training Images")
            ax1.set_title("Images per Character")
            ax2.barh(names, sizes_mb, color="#e15759")
            ax2.set_xlabel("Size (MB)")
            ax2.set_title("LoRA File Size")
            plt.tight_layout()
            plt.show()

In [ ]:
#@title 7b. Export All LoRAs as ZIP
#@markdown Creates a downloadable zip of all trained character LoRAs.

import os, json, zipfile
from pathlib import Path
from datetime import datetime

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
lora_root = WAREHOUSE / "lora"
export_dir = WAREHOUSE / "exports"
export_dir.mkdir(parents=True, exist_ok=True)

characters = [d for d in sorted(lora_root.iterdir()) if d.is_dir()] if lora_root.exists() else []

if not characters:
    print("No trained LoRAs to export.")
else:
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    zip_path = export_dir / f"aniloom_loras_{ts}.zip"

    manifest = {"exported": datetime.now().isoformat(), "characters": []}

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for char_dir in characters:
            meta_path = char_dir / "metadata.json"
            if meta_path.exists():
                manifest["characters"].append(json.loads(meta_path.read_text()))

            for f in char_dir.iterdir():
                zf.write(f, f"lora/{char_dir.name}/{f.name}")
                print(f"  Added {char_dir.name}/{f.name}")

        # Write manifest inside zip
        zf.writestr("manifest.json", json.dumps(manifest, indent=2))

    size_mb = zip_path.stat().st_size / 1e6
    print(f"\n✅ Export: {zip_path}  ({size_mb:.1f} MB)")
    print(f"   {len(manifest['characters'])} characters packaged.")
    print("\nTo download: right-click the file in the Drive sidebar → Download")

## 8. Colab Survival Mode
Keep the session alive during long training runs.

In [ ]:
#@title 8. Enable Colab Survival (optional)
#@markdown Prevents Colab from disconnecting during long training.
#@markdown Checkpoints every 5 minutes to Drive.

import os, sys
sys.path.insert(0, "/content/AnimeLoom")

from cloud.colab_survival import ColabSurvival

survival = ColabSurvival(
    warehouse_path=os.environ["AI_CACHE_ROOT"],
    keepalive_interval=240,   # ping every 4 min
    checkpoint_interval=300,  # save every 5 min
)
survival.start()
print("Survival daemon running — keepalive 4min, checkpoint 5min.")
print("Your work is auto-saved to Google Drive.")

## 9. Anime Video Generation (SDXL + CogVideoX)
Generates anime-style video with **real motion** — smiling, expressions, head movement, hair swaying.

| Stage | What | Purpose |
|-------|------|---------|
| **SDXL + LoRA** | High-res keyframes | Character identity locked in via your trained LoRA |
| **CogVideoX-5B** | Image-to-video | Turns each keyframe into 49 frames of actual anime motion |
| **Stitch** | Cross-fade + assemble | Joins motion clips into a continuous video |

Each model loads/unloads sequentially to fit on T4 (16GB VRAM). CogVideoX uses int8 quantization to stay under 8GB.

In [ ]:
#@title 9. Generate Anime Video (SDXL + CogVideoX 1.5)
#@markdown Full-body anime motion — expressions, movement, head to toe.
#@markdown Uses SDXL LoRA for character identity + CogVideoX 1.5 for anime-style video.
#@markdown **Requires A100 GPU** (Runtime → Change runtime type → A100).

CHARACTER_NAME = "denji"  #@param {type:"string"}
NEGATIVE_PROMPT = "blurry, low quality, deformed, extra fingers, worst quality, ugly, disfigured, bad anatomy, bad hands, 3d render, cgi, photorealistic, live action"  #@param {type:"string"}
DENOISING_STRENGTH = 0.45  #@param {type:"slider", min:0.3, max:0.8, step:0.05}
IMAGE_WIDTH = 512  #@param {type:"slider", min:512, max:1024, step:128}
IMAGE_HEIGHT = 768  #@param {type:"slider", min:512, max:1024, step:128}
LORA_SCALE = 1.0  #@param {type:"slider", min:0.5, max:2.0, step:0.1}
SDXL_GUIDANCE = 7.5  #@param {type:"slider", min:1.0, max:15.0, step:0.5}
SDXL_STEPS = 30  #@param {type:"slider", min:10, max:50, step:5}

#@markdown ---
#@markdown **CogVideoX Motion Settings**
NUM_FRAMES = 49  #@param {type:"slider", min:13, max:49, step:12}
COGVID_STEPS = 60  #@param {type:"slider", min:20, max:100, step:5}
COGVID_GUIDANCE = 7.5  #@param {type:"slider", min:1.0, max:12.0, step:0.5}
USE_DYNAMIC_CFG = True  #@param {type:"boolean"}
FPS = 16  #@param {type:"slider", min:8, max:24, step:4}

#@markdown ---
#@markdown **Face Restoration** (fixes blurry faces during motion)
FACE_RESTORE = True  #@param {type:"boolean"}

#@markdown **Keyframe prompts** — one per line. Each becomes a keyframe, then CogVideoX animates it.
KEYFRAME_PROMPTS = """denji, 1boy, full body, head to toe, facing viewer, front view, detailed face, detailed hair strands, symmetrical eyes, standing, looking at viewer, blonde messy hair, sharp teeth, school uniform, anime screencap, studio quality, sharp lineart, vibrant colors, masterpiece, best quality
denji, 1boy, full body, head to toe, facing viewer, front view, detailed face, detailed hair strands, symmetrical eyes, standing, slight grin, blonde messy hair, sharp teeth, school uniform, anime screencap, studio quality, sharp lineart, vibrant colors, masterpiece, best quality
denji, 1boy, full body, head to toe, facing viewer, front view, detailed face, detailed hair strands, symmetrical eyes, standing, looking to the side, blonde messy hair, school uniform, anime screencap, studio quality, sharp lineart, vibrant colors, masterpiece, best quality
denji, 1boy, full body, head to toe, facing viewer, front view, detailed face, detailed hair strands, symmetrical eyes, standing, serious expression, blonde messy hair, school uniform, anime screencap, studio quality, sharp lineart, vibrant colors, masterpiece, best quality
denji, 1boy, full body, head to toe, facing viewer, front view, detailed face, detailed hair strands, symmetrical eyes, standing, relaxed pose, blonde messy hair, school uniform, anime screencap, studio quality, sharp lineart, vibrant colors, masterpiece, best quality
denji, 1boy, full body, head to toe, facing viewer, front view, detailed face, detailed hair strands, symmetrical eyes, standing, confident smirk, blonde messy hair, sharp teeth, school uniform, anime screencap, studio quality, sharp lineart, vibrant colors, masterpiece, best quality"""  #@param {type:"string"}

#@markdown **Motion prompts** — describe the motion for CogVideoX. One per keyframe (cycles if fewer).
MOTION_PROMPTS = """anime boy standing, full body visible, subtle weight shift, slow deliberate blink, hair gently swaying each strand clearly defined, warm lighting, detailed face, sharp facial features, clear expression, symmetrical eyes, stable eye shape, clear detailed eyes throughout, consistent pose, stable camera angle
anime boy standing, full body visible, slight grin, soft head tilt, calm atmosphere, detailed face, sharp facial features, clear expression, symmetrical eyes, stable eye shape, clear detailed eyes throughout, consistent pose, stable camera angle
anime boy standing, full body visible, glancing to the side, hair gently swaying each strand clearly defined, serene expression, detailed face, sharp facial features, clear expression, symmetrical eyes, stable eye shape, clear detailed eyes throughout, consistent pose, stable camera angle
anime boy standing, full body visible, serious focused expression, slight eye movement, dramatic lighting, detailed face, sharp facial features, clear expression, symmetrical eyes, stable eye shape, clear detailed eyes throughout, consistent pose, stable camera angle
anime boy standing, full body visible, relaxed posture, subtle breathing, soft light, detailed face, sharp facial features, clear expression, symmetrical eyes, stable eye shape, consistent pose, stable camera angle
anime boy standing, full body visible, confident smirk, looking directly at viewer, determined expression, detailed face, sharp facial features, clear expression, symmetrical eyes, stable eye shape, clear detailed eyes throughout, consistent pose, stable camera angle"""  #@param {type:"string"}

#@markdown **Motion negative prompt** — steers CogVideoX away from blur artifacts.
MOTION_NEGATIVE = "blurry face, distorted mouth, mouth blur, facial blur, inconsistent facial expression, fuzzy features, low quality, worst quality, deformed face, blurry hair, undefined hair strands, camera shake, shifting perspective, rotating view, distorted eyes, asymmetric eyes, warped eyes, misaligned pupils, blurry eyes, unfocused eyes, 3d render, photorealistic"  #@param {type:"string"}

import os, gc, sys, time, torch, json, subprocess
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
char_id = CHARACTER_NAME.lower().replace(" ", "_")

kf_prompts = [p.strip() for p in KEYFRAME_PROMPTS.strip().split("\n") if p.strip()]
motion_prompts = [p.strip() for p in MOTION_PROMPTS.strip().split("\n") if p.strip()]
num_kf = len(kf_prompts)

total_frames = num_kf * NUM_FRAMES
print(f"Plan: {num_kf} keyframes × {NUM_FRAMES} CogVideoX frames = {total_frames} frames ({total_frames/FPS:.1f}s at {FPS}fps)")
print(f"Model: CogVideoX 1.5 | Steps: {COGVID_STEPS} | Guidance: {COGVID_GUIDANCE} | Face restore: {FACE_RESTORE}")
print(f"SDXL output: {IMAGE_WIDTH}×{IMAGE_HEIGHT} (portrait) → CogVideoX input: 480×720")
print("=" * 60)

# ================================================================
# Phase 1: SDXL + LoRA — Generate keyframes with character identity
# ================================================================
gc.collect()
torch.cuda.empty_cache()

lora_dir = WAREHOUSE / "lora" / char_id
meta_path = lora_dir / "metadata.json"
if not meta_path.exists():
    raise FileNotFoundError(f"No SDXL LoRA for '{CHARACTER_NAME}' at {lora_dir}. Run step 4 first.")

meta = json.loads(meta_path.read_text())
base_model = meta["base_model"]
print(f"Loading {base_model}…")

from diffusers import StableDiffusionXLPipeline, AutoPipelineForImage2Image, DPMSolverMultistepScheduler
from peft import PeftModel

pipe = StableDiffusionXLPipeline.from_pretrained(base_model, torch_dtype=torch.float16)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.unet = PeftModel.from_pretrained(pipe.unet, str(lora_dir))
pipe.unet.eval()

if LORA_SCALE != 1.0:
    for module in pipe.unet.modules():
        if hasattr(module, "scaling"):
            for key in module.scaling:
                module.scaling[key] = LORA_SCALE

pipe.to("cuda")
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()
try:
    pipe.enable_xformers_memory_efficient_attention()
except Exception:
    pass

pipe_i2i = AutoPipelineForImage2Image.from_pipe(pipe)
print(f"SDXL + LoRA loaded (scale={LORA_SCALE}).")

print(f"\nGenerating {num_kf} keyframes…")
keyframes = []

for i, prompt in enumerate(kf_prompts):
    gen = torch.Generator("cuda").manual_seed(42 + i)

    if i == 0:
        result = pipe(
            prompt=prompt,
            negative_prompt=NEGATIVE_PROMPT,
            height=IMAGE_HEIGHT, width=IMAGE_WIDTH,
            num_inference_steps=SDXL_STEPS,
            guidance_scale=SDXL_GUIDANCE,
            generator=gen,
        )
    else:
        result = pipe_i2i(
            prompt=prompt,
            negative_prompt=NEGATIVE_PROMPT,
            image=keyframes[-1],
            strength=DENOISING_STRENGTH,
            num_inference_steps=SDXL_STEPS,
            guidance_scale=SDXL_GUIDANCE,
            generator=gen,
        )

    keyframes.append(result.images[0])
    print(f"  Keyframe {i+1}/{num_kf} done")

kf_dir = WAREHOUSE / "outputs" / f"keyframes_{char_id}"
kf_dir.mkdir(parents=True, exist_ok=True)
for i, kf in enumerate(keyframes):
    kf.save(kf_dir / f"kf_{i:03d}.png")
print(f"Keyframes saved to {kf_dir}")

# Unload SDXL
while hasattr(pipe.unet, "base_model"):
    try:
        pipe.unet = pipe.unet.base_model.model
    except Exception:
        break
del pipe, pipe_i2i
gc.collect()
torch.cuda.empty_cache()
print("SDXL unloaded.\n")

# ================================================================
# Phase 2: CogVideoX 1.5 — Animate each keyframe into real motion
# ================================================================
print("Loading CogVideoX 1.5 image-to-video (int8 quantized)…")

from diffusers import CogVideoXImageToVideoPipeline
from optimum.quanto import qint8, quantize, freeze

cogvid_pipe = CogVideoXImageToVideoPipeline.from_pretrained(
    "THUDM/CogVideoX1.5-5B-I2V",
    torch_dtype=torch.bfloat16,
)

# Int8 quantization — reduces VRAM usage
quantize(cogvid_pipe.transformer, weights=qint8)
freeze(cogvid_pipe.transformer)
quantize(cogvid_pipe.text_encoder, weights=qint8)
freeze(cogvid_pipe.text_encoder)

cogvid_pipe.enable_model_cpu_offload()
cogvid_pipe.vae.enable_tiling()
cogvid_pipe.vae.enable_slicing()
print("CogVideoX 1.5 loaded and quantized.")

# Generate motion clips
all_clips = []
start_time = time.time()

for i, kf_image in enumerate(keyframes):
    motion_prompt = motion_prompts[i % len(motion_prompts)]
    print(f"\n  Animating keyframe {i+1}/{num_kf}: \"{motion_prompt[:60]}…\"")

    # Aspect-ratio-preserving resize for CogVideoX (portrait: 480×720)
    kf_w, kf_h = kf_image.size
    target_ratio = 480 / 720  # 0.667 (portrait)
    current_ratio = kf_w / kf_h
    if current_ratio > target_ratio:
        new_w = int(kf_h * target_ratio)
        left = (kf_w - new_w) // 2
        kf_cropped = kf_image.crop((left, 0, left + new_w, kf_h))
    else:
        new_h = int(kf_w / target_ratio)
        top = (kf_h - new_h) // 2
        kf_cropped = kf_image.crop((0, top, kf_w, top + new_h))
    kf_resized = kf_cropped.resize((480, 720), Image.LANCZOS)

    gen = torch.Generator("cpu").manual_seed(42 + i)

    output = cogvid_pipe(
        image=kf_resized,
        prompt=motion_prompt,
        negative_prompt=MOTION_NEGATIVE,
        num_frames=NUM_FRAMES,
        num_inference_steps=COGVID_STEPS,
        guidance_scale=COGVID_GUIDANCE,
        use_dynamic_cfg=USE_DYNAMIC_CFG,
        generator=gen,
    )

    clip_frames = output.frames[0]  # list of PIL Images
    all_clips.append(clip_frames)

    elapsed = time.time() - start_time
    eta = (elapsed / (i + 1)) * (num_kf - i - 1)
    print(f"  Clip {i+1}/{num_kf} done — {len(clip_frames)} frames ({elapsed:.0f}s elapsed, ~{eta:.0f}s remaining)")

    gc.collect()
    torch.cuda.empty_cache()

cogvid_time = time.time() - start_time
print(f"\nCogVideoX done in {cogvid_time/60:.1f} minutes.")

# Unload CogVideoX
del cogvid_pipe
gc.collect()
torch.cuda.empty_cache()
print("CogVideoX unloaded.\n")

# ================================================================
# Phase 3: Face restoration + frame sharpening
# ================================================================
import cv2

if FACE_RESTORE:
    # --- GFPGAN: restore faces ---
    print("Loading GFPGAN face restorer…")
    try:
        from gfpgan import GFPGANer

        restorer = GFPGANer(
            model_path="https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth",
            upscale=1,
            arch="clean",
            channel_multiplier=2,
            bg_upsampler=None,
        )
        print("GFPGAN loaded. Restoring faces…")

        restore_start = time.time()
        for clip_idx, clip in enumerate(all_clips):
            restored_clip = []
            for frame_pil in clip:
                frame_bgr = cv2.cvtColor(np.array(frame_pil), cv2.COLOR_RGB2BGR)
                _, _, restored_bgr = restorer.enhance(
                    frame_bgr, has_aligned=False, only_center_face=False, paste_back=True
                )
                restored_rgb = cv2.cvtColor(restored_bgr, cv2.COLOR_BGR2RGB)
                restored_clip.append(Image.fromarray(restored_rgb))
            all_clips[clip_idx] = restored_clip

        restore_time = time.time() - restore_start
        print(f"Face restoration done in {restore_time:.0f}s.")

        del restorer
        gc.collect()
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"Face restoration failed ({e}) — using raw CogVideoX output.")

    # --- Real-ESRGAN: sharpen entire frame (fixes blurry hair) ---
    print("Loading Real-ESRGAN anime sharpener…")
    try:
        from basicsr.archs.rrdbnet_arch import RRDBNet
        from realesrgan import RealESRGANer

        esrgan_model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=6, num_grow_ch=32, scale=4)
        esrgan = RealESRGANer(
            scale=4,
            model_path="https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth",
            dni_weight=None,
            model=esrgan_model,
            half=True,
        )
        print("Real-ESRGAN loaded. Sharpening frames…")

        sharpen_start = time.time()
        for clip_idx, clip in enumerate(all_clips):
            sharpened_clip = []
            for frame_pil in clip:
                frame_bgr = cv2.cvtColor(np.array(frame_pil), cv2.COLOR_RGB2BGR)
                h_orig, w_orig = frame_bgr.shape[:2]
                output_bgr, _ = esrgan.enhance(frame_bgr, outscale=1)
                if output_bgr.shape[:2] != (h_orig, w_orig):
                    output_bgr = cv2.resize(output_bgr, (w_orig, h_orig), interpolation=cv2.INTER_LANCZOS4)
                output_rgb = cv2.cvtColor(output_bgr, cv2.COLOR_BGR2RGB)
                sharpened_clip.append(Image.fromarray(output_rgb))
            all_clips[clip_idx] = sharpened_clip

        sharpen_time = time.time() - sharpen_start
        print(f"Frame sharpening done in {sharpen_time:.0f}s.")

        del esrgan, esrgan_model
        gc.collect()
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"Real-ESRGAN sharpening failed ({e}) — skipping.")

# ================================================================
# Phase 4: Stitch clips with cross-fade transitions + save video
# ================================================================
CROSSFADE_FRAMES = 4

print(f"Stitching {len(all_clips)} clips (cross-fade={CROSSFADE_FRAMES} frames)…")

all_frames = []
for clip_idx, clip in enumerate(all_clips):
    clip_arrays = [np.array(frame) for frame in clip]

    if clip_idx == 0:
        all_frames.extend(clip_arrays)
    else:
        overlap = min(CROSSFADE_FRAMES, len(all_frames), len(clip_arrays))
        for j in range(overlap):
            t = (j + 1) / (overlap + 1)
            prev_frame = all_frames[-(overlap - j)].astype(np.float32)
            next_frame = clip_arrays[j].astype(np.float32)
            blended = ((1 - t) * prev_frame + t * next_frame).clip(0, 255).astype(np.uint8)
            all_frames[-(overlap - j)] = blended
        all_frames.extend(clip_arrays[overlap:])

print(f"Total: {len(all_frames)} frames ({len(all_frames)/FPS:.1f}s at {FPS}fps)")

# Save video
output_dir = WAREHOUSE / "outputs" / f"cogvid_{char_id}"
output_dir.mkdir(parents=True, exist_ok=True)
video_path = output_dir / f"{char_id}_clip.mp4"

h, w = all_frames[0].shape[:2]
writer = cv2.VideoWriter(str(video_path), cv2.VideoWriter_fourcc(*"mp4v"), FPS, (w, h))
for frame in all_frames:
    writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
writer.release()

duration = len(all_frames) / FPS
size_mb = video_path.stat().st_size / 1e6
total_time = time.time() - start_time
print(f"\nVideo saved: {video_path}")
print(f"Duration: {duration:.1f}s | Frames: {len(all_frames)} | Size: {size_mb:.1f}MB")
print(f"Total time: {total_time/60:.1f} minutes")

# Show keyframe preview
cols = min(num_kf, 6)
rows = (num_kf + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
axes_flat = np.array(axes).flat if hasattr(axes, 'flat') else [axes]
for i, (ax, kf) in enumerate(zip(axes_flat, keyframes)):
    ax.imshow(kf)
    ax.set_title(f"KF {i}", fontsize=9)
    ax.axis("off")
for j in range(i + 1, rows * cols):
    axes_flat[j].axis("off")
plt.suptitle(f"'{CHARACTER_NAME}' — SDXL keyframes (animated by CogVideoX 1.5)", fontsize=13)
plt.tight_layout()
plt.show()

gc.collect()
torch.cuda.empty_cache()
print("\nDone. VRAM freed.")

## 10. Generate 2-Minute Long Anime Video
Same CogVideoX pipeline as step 9, but scaled up. Generates many SDXL keyframes and animates each into a motion clip, then stitches everything into a long continuous anime video. Run step 9 first to verify quality.

In [ ]:
#@title 10. Generate 2-Minute Long Anime Video (SDXL + CogVideoX 1.5)
#@markdown Generates a long full-body anime video with real motion. Run step 9 first to verify quality.
#@markdown **Requires A100 GPU.** ~60-120 min depending on target duration.

CHARACTER_NAME = "denji"  #@param {type:"string"}
TARGET_SECONDS = 120  #@param {type:"slider", min:30, max:300, step:30}
FPS = 16  #@param {type:"slider", min:8, max:24, step:4}
DENOISING_STRENGTH = 0.40  #@param {type:"slider", min:0.3, max:0.8, step:0.05}
IMAGE_WIDTH = 512  #@param {type:"slider", min:512, max:1024, step:128}
IMAGE_HEIGHT = 768  #@param {type:"slider", min:512, max:1024, step:128}
LORA_SCALE = 1.0  #@param {type:"slider", min:0.5, max:2.0, step:0.1}
SDXL_GUIDANCE = 7.5  #@param {type:"slider", min:1.0, max:15.0, step:0.5}
SDXL_STEPS = 25  #@param {type:"slider", min:10, max:50, step:5}

#@markdown ---
#@markdown **CogVideoX Settings**
NUM_FRAMES = 49  #@param {type:"slider", min:13, max:49, step:12}
COGVID_STEPS = 60  #@param {type:"slider", min:20, max:100, step:5}
COGVID_GUIDANCE = 7.5  #@param {type:"slider", min:1.0, max:12.0, step:0.5}
USE_DYNAMIC_CFG = True  #@param {type:"boolean"}

#@markdown ---
#@markdown **Face Restoration**
FACE_RESTORE = True  #@param {type:"boolean"}

#@markdown Scene prompts — the pipeline cycles through these for variety.
SCENE_PROMPTS = """denji, 1boy, full body, head to toe, facing viewer, front view, detailed face, detailed hair strands, symmetrical eyes, walking through city street, blonde messy hair, sharp teeth, anime screencap, studio quality, sharp lineart, vibrant colors, masterpiece, best quality
denji, 1boy, full body, head to toe, facing viewer, front view, detailed face, detailed hair strands, symmetrical eyes, standing eating bread, happy expression, blonde messy hair, anime screencap, studio quality, sharp lineart, vibrant colors, masterpiece, best quality
denji, 1boy, full body, head to toe, facing viewer, front view, detailed face, detailed hair strands, symmetrical eyes, standing on rooftop, sunset light, blonde messy hair, anime screencap, studio quality, sharp lineart, vibrant colors, masterpiece, best quality
denji, 1boy, full body, head to toe, facing viewer, front view, detailed face, detailed hair strands, symmetrical eyes, walking through alley, dramatic lighting, blonde messy hair, anime screencap, studio quality, sharp lineart, vibrant colors, masterpiece, best quality"""  #@param {type:"string"}

#@markdown Motion prompts — describe the animation per scene.
MOTION_PROMPTS = """anime boy walking forward on city street, full body visible, steady footsteps, looking around, hair gently swaying each strand clearly defined, detailed face, sharp facial features, clear expression, symmetrical eyes, stable eye shape, clear detailed eyes throughout, consistent pose, stable camera angle
anime boy standing, full body visible, eating happily, joyful expression, slow deliberate blink, detailed face, sharp facial features, clear expression, symmetrical eyes, stable eye shape, clear detailed eyes throughout, consistent pose, stable camera angle
anime boy standing on rooftop, full body visible, gazing at sunset, hair gently swaying each strand clearly defined, detailed face, sharp facial features, clear expression, symmetrical eyes, stable eye shape, clear detailed eyes throughout, consistent pose, stable camera angle
anime boy walking through alley, full body visible, cautious expression, dramatic shadows, slow deliberate blink, detailed face, sharp facial features, clear expression, symmetrical eyes, stable eye shape, clear detailed eyes throughout, consistent pose, stable camera angle"""  #@param {type:"string"}

NEGATIVE_PROMPT = "blurry, low quality, deformed, extra fingers, worst quality, ugly, disfigured, bad anatomy, bad hands, 3d render, cgi, photorealistic, live action"  #@param {type:"string"}
MOTION_NEGATIVE = "blurry face, distorted mouth, mouth blur, facial blur, inconsistent facial expression, fuzzy features, low quality, worst quality, deformed face, blurry hair, undefined hair strands, camera shake, shifting perspective, rotating view, distorted eyes, asymmetric eyes, warped eyes, misaligned pupils, blurry eyes, unfocused eyes, 3d render, photorealistic"  #@param {type:"string"}

import os, gc, sys, time, torch, json, subprocess
import numpy as np
from pathlib import Path
from PIL import Image

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
char_id = CHARACTER_NAME.lower().replace(" ", "_")
scenes = [s.strip() for s in SCENE_PROMPTS.strip().split("\n") if s.strip()]
motions = [s.strip() for s in MOTION_PROMPTS.strip().split("\n") if s.strip()]

# Calculate keyframes needed
CROSSFADE_FRAMES = 4
effective_frames_per_clip = NUM_FRAMES - CROSSFADE_FRAMES
total_frames_needed = TARGET_SECONDS * FPS
num_keyframes = max(2, (total_frames_needed + effective_frames_per_clip - 1) // effective_frames_per_clip)

total_frames = num_keyframes * NUM_FRAMES - (num_keyframes - 1) * CROSSFADE_FRAMES
actual_duration = total_frames / FPS

print(f"Target: {TARGET_SECONDS}s at {FPS}fps")
print(f"Keyframes needed: {num_keyframes}")
print(f"After CogVideoX 1.5 + stitch: ~{total_frames} frames ({actual_duration:.1f}s)")
print(f"Model: CogVideoX 1.5 | Steps: {COGVID_STEPS} | Guidance: {COGVID_GUIDANCE} | Face restore: {FACE_RESTORE}")
print(f"SDXL output: {IMAGE_WIDTH}×{IMAGE_HEIGHT} (portrait) → CogVideoX input: 480×720")
print("=" * 60)

# ================================================================
# Phase 1: SDXL + LoRA — Generate keyframes
# ================================================================
gc.collect()
torch.cuda.empty_cache()

lora_dir = WAREHOUSE / "lora" / char_id
meta_path = lora_dir / "metadata.json"
if not meta_path.exists():
    raise FileNotFoundError(f"No SDXL LoRA for '{CHARACTER_NAME}'. Run step 4 first.")

meta = json.loads(meta_path.read_text())
base_model = meta["base_model"]
print(f"\nLoading {base_model}…")

from diffusers import StableDiffusionXLPipeline, AutoPipelineForImage2Image, DPMSolverMultistepScheduler
from peft import PeftModel

pipe = StableDiffusionXLPipeline.from_pretrained(base_model, torch_dtype=torch.float16)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.unet = PeftModel.from_pretrained(pipe.unet, str(lora_dir))
pipe.unet.eval()

if LORA_SCALE != 1.0:
    for module in pipe.unet.modules():
        if hasattr(module, "scaling"):
            for key in module.scaling:
                module.scaling[key] = LORA_SCALE

pipe.to("cuda")
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()
try:
    pipe.enable_xformers_memory_efficient_attention()
except Exception:
    pass

pipe_i2i = AutoPipelineForImage2Image.from_pipe(pipe)
print("SDXL + LoRA loaded.")

print(f"\nGenerating {num_keyframes} keyframes…")
keyframes = []
kf_start = time.time()

for i in range(num_keyframes):
    prompt = scenes[i % len(scenes)]
    gen = torch.Generator("cuda").manual_seed(42 + i * 3)

    if i == 0:
        result = pipe(
            prompt=prompt,
            negative_prompt=NEGATIVE_PROMPT,
            height=IMAGE_HEIGHT, width=IMAGE_WIDTH,
            num_inference_steps=SDXL_STEPS,
            guidance_scale=SDXL_GUIDANCE,
            generator=gen,
        )
    else:
        result = pipe_i2i(
            prompt=prompt,
            negative_prompt=NEGATIVE_PROMPT,
            image=keyframes[-1],
            strength=DENOISING_STRENGTH,
            num_inference_steps=SDXL_STEPS,
            guidance_scale=SDXL_GUIDANCE,
            generator=gen,
        )

    keyframes.append(result.images[0])

    elapsed = time.time() - kf_start
    eta = (elapsed / (i + 1)) * (num_keyframes - i - 1)
    print(f"  KF {i+1}/{num_keyframes} done ({elapsed:.0f}s elapsed, ~{eta:.0f}s remaining)")

    if i % 20 == 19:
        gc.collect()
        torch.cuda.empty_cache()

kf_time = time.time() - kf_start
print(f"Keyframes generated in {kf_time/60:.1f} minutes.")

# Save keyframes
kf_dir = WAREHOUSE / "outputs" / f"keyframes_{char_id}_long"
kf_dir.mkdir(parents=True, exist_ok=True)
for i, kf in enumerate(keyframes):
    kf.save(kf_dir / f"kf_{i:04d}.png")

# Unload SDXL
while hasattr(pipe.unet, "base_model"):
    try:
        pipe.unet = pipe.unet.base_model.model
    except Exception:
        break
del pipe, pipe_i2i
gc.collect()
torch.cuda.empty_cache()
print("SDXL unloaded.\n")

# ================================================================
# Phase 2: CogVideoX 1.5 — Animate each keyframe
# ================================================================
print("Loading CogVideoX 1.5 image-to-video (int8 quantized)…")

from diffusers import CogVideoXImageToVideoPipeline
from optimum.quanto import qint8, quantize, freeze

cogvid_pipe = CogVideoXImageToVideoPipeline.from_pretrained(
    "THUDM/CogVideoX1.5-5B-I2V",
    torch_dtype=torch.bfloat16,
)

quantize(cogvid_pipe.transformer, weights=qint8)
freeze(cogvid_pipe.transformer)
quantize(cogvid_pipe.text_encoder, weights=qint8)
freeze(cogvid_pipe.text_encoder)

cogvid_pipe.enable_model_cpu_offload()
cogvid_pipe.vae.enable_tiling()
cogvid_pipe.vae.enable_slicing()
print("CogVideoX 1.5 loaded and quantized.")

# Setup face restorer + frame sharpener if enabled
import cv2
face_restorer = None
frame_sharpener = None

if FACE_RESTORE:
    try:
        from gfpgan import GFPGANer
        face_restorer = GFPGANer(
            model_path="https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth",
            upscale=1,
            arch="clean",
            channel_multiplier=2,
            bg_upsampler=None,
        )
        print("GFPGAN face restorer loaded.")
    except Exception as e:
        print(f"Face restoration unavailable ({e}) — skipping.")

    try:
        from basicsr.archs.rrdbnet_arch import RRDBNet
        from realesrgan import RealESRGANer

        esrgan_model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=6, num_grow_ch=32, scale=4)
        frame_sharpener = RealESRGANer(
            scale=4,
            model_path="https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth",
            dni_weight=None,
            model=esrgan_model,
            half=True,
        )
        print("Real-ESRGAN anime sharpener loaded.")
    except Exception as e:
        print(f"Real-ESRGAN unavailable ({e}) — skipping.")

# Process keyframes and write frames to disk to save RAM
frames_dir = WAREHOUSE / "outputs" / f"frames_{char_id}_long"
frames_dir.mkdir(parents=True, exist_ok=True)

frame_count = 0
cogvid_start = time.time()
prev_tail_frames = None

for i, kf_image in enumerate(keyframes):
    motion_prompt = motions[i % len(motions)]
    print(f"\n  Animating KF {i+1}/{num_keyframes}: \"{motion_prompt[:50]}…\"")

    # Aspect-ratio-preserving resize for CogVideoX (portrait: 480×720)
    kf_w, kf_h = kf_image.size
    target_ratio = 480 / 720  # 0.667 (portrait)
    current_ratio = kf_w / kf_h
    if current_ratio > target_ratio:
        new_w = int(kf_h * target_ratio)
        left = (kf_w - new_w) // 2
        kf_cropped = kf_image.crop((left, 0, left + new_w, kf_h))
    else:
        new_h = int(kf_w / target_ratio)
        top = (kf_h - new_h) // 2
        kf_cropped = kf_image.crop((0, top, kf_w, top + new_h))
    kf_resized = kf_cropped.resize((480, 720), Image.LANCZOS)

    gen = torch.Generator("cpu").manual_seed(42 + i)

    output = cogvid_pipe(
        image=kf_resized,
        prompt=motion_prompt,
        negative_prompt=MOTION_NEGATIVE,
        num_frames=NUM_FRAMES,
        num_inference_steps=COGVID_STEPS,
        guidance_scale=COGVID_GUIDANCE,
        use_dynamic_cfg=USE_DYNAMIC_CFG,
        generator=gen,
    )

    clip_frames = [np.array(f) for f in output.frames[0]]

    # Face restoration per frame
    if face_restorer is not None:
        restored = []
        for frame in clip_frames:
            frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
            _, _, out_bgr = face_restorer.enhance(frame_bgr, has_aligned=False, only_center_face=False, paste_back=True)
            restored.append(cv2.cvtColor(out_bgr, cv2.COLOR_BGR2RGB))
        clip_frames = restored

    # Real-ESRGAN sharpening per frame (fixes blurry hair)
    if frame_sharpener is not None:
        sharpened = []
        for frame in clip_frames:
            frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
            h_orig, w_orig = frame_bgr.shape[:2]
            out_bgr, _ = frame_sharpener.enhance(frame_bgr, outscale=1)
            if out_bgr.shape[:2] != (h_orig, w_orig):
                out_bgr = cv2.resize(out_bgr, (w_orig, h_orig), interpolation=cv2.INTER_LANCZOS4)
            sharpened.append(cv2.cvtColor(out_bgr, cv2.COLOR_BGR2RGB))
        clip_frames = sharpened

    if prev_tail_frames is not None:
        overlap = min(CROSSFADE_FRAMES, len(prev_tail_frames), len(clip_frames))
        for j in range(overlap):
            t = (j + 1) / (overlap + 1)
            prev_f = prev_tail_frames[-(overlap - j)].astype(np.float32)
            next_f = clip_frames[j].astype(np.float32)
            blended = ((1 - t) * prev_f + t * next_f).clip(0, 255).astype(np.uint8)
            overwrite_idx = frame_count - (overlap - j)
            Image.fromarray(blended).save(frames_dir / f"frame_{overwrite_idx:06d}.png")
        for f in clip_frames[overlap:]:
            Image.fromarray(f).save(frames_dir / f"frame_{frame_count:06d}.png")
            frame_count += 1
    else:
        for f in clip_frames:
            Image.fromarray(f).save(frames_dir / f"frame_{frame_count:06d}.png")
            frame_count += 1

    prev_tail_frames = clip_frames[-CROSSFADE_FRAMES:]

    elapsed = time.time() - cogvid_start
    eta = (elapsed / (i + 1)) * (num_keyframes - i - 1)
    print(f"  Clip {i+1}/{num_keyframes} done — {len(clip_frames)} frames ({elapsed:.0f}s elapsed, ~{eta/60:.0f}m remaining)")

    gc.collect()
    torch.cuda.empty_cache()

cogvid_time = time.time() - cogvid_start
print(f"\nCogVideoX done in {cogvid_time/60:.1f} minutes. {frame_count} frames saved to disk.")

del cogvid_pipe
if face_restorer is not None:
    del face_restorer
if frame_sharpener is not None:
    del frame_sharpener
gc.collect()
torch.cuda.empty_cache()

# ================================================================
# Phase 3: Assemble video from saved frames
# ================================================================
print(f"\nAssembling video from {frame_count} frames…")

output_dir = WAREHOUSE / "outputs" / "long_video"
output_dir.mkdir(parents=True, exist_ok=True)
video_path = output_dir / f"{char_id}_{TARGET_SECONDS}s.mp4"

first_frame = np.array(Image.open(frames_dir / "frame_000000.png"))
h, w = first_frame.shape[:2]

writer = cv2.VideoWriter(str(video_path), cv2.VideoWriter_fourcc(*"mp4v"), FPS, (w, h))
for idx in range(frame_count):
    frame_path = frames_dir / f"frame_{idx:06d}.png"
    if frame_path.exists():
        frame = np.array(Image.open(frame_path))
        writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
writer.release()

duration = frame_count / FPS
size_mb = video_path.stat().st_size / 1e6
total_time = time.time() - kf_start

print(f"\nVideo saved: {video_path}")
print(f"Duration: {duration:.1f}s | Frames: {frame_count} | Size: {size_mb:.1f}MB")
print(f"Total time: {total_time/60:.1f} minutes (keyframes: {kf_time/60:.1f}m, CogVideoX: {cogvid_time/60:.1f}m)")

gc.collect()
torch.cuda.empty_cache()
print("Done. VRAM freed.")

## 10b. Text-to-Anime — Sora-like Generation (NEW)
Type a **natural-language story** and get an anime video. No need to write prompts manually.

| Stage | What | Purpose |
|-------|------|---------|
| **Story Decomposer** | Gemini Flash (free) or local NLP | Converts your text into SCENE/CHAR shots |
| **SDXL + LoRA** | Character reference keyframes | Locks character identity via trained LoRA |
| **CogVideoX 1.5** | Image-to-video animation | Turns each keyframe into 49 frames of motion |
| **RIFE** | Temporal upscaling | 8fps → 24fps smooth animation |
| **Real-ESRGAN** | Spatial upscaling | 480p → 720p+ resolution |
| **Color Grading** | Anime colour enhancement | Vibrant, studio-quality colour |
| **Assembly** | Cross-dissolve stitch | Smooth transitions between shots |

**Requirements:**
- Run Step 1 (Setup) first
- For character consistency: train LoRAs in Steps 4-6 first
- **Gemini API key** (optional, free): Get one at [aistudio.google.com](https://aistudio.google.com/apikey)
- Works on **T4** (free tier) or **A100** (Pro)

In [ ]:
#@title 10b. Text-to-Anime — Sora-like Generation
#@markdown Type a story and get an anime video. Uses the full AnimeLoom pipeline:
#@markdown Story Decomposer → SDXL+LoRA keyframes → AnimateDiff+LoRA motion → RIFE upscale → Real-ESRGAN → Color grade → Assembly.

#@markdown ---
#@markdown **Your Story** (natural language — describe what happens)
STORY_TEXT = "A girl with long blue hair walks through a cherry blossom forest at sunset. Petals fall around her as she stops at a wooden bridge and looks at the river below. The wind gently moves her hair."  #@param {type:"string"}

#@markdown ---
#@markdown **Quality Preset**
QUALITY = "standard"  #@param ["draft", "standard", "high"]

#@markdown ---
#@markdown **Gemini API Key** (optional — get free at aistudio.google.com/apikey)
GEMINI_API_KEY = ""  #@param {type:"string"}

#@markdown ---
#@markdown **Character Name** (must have trained LoRA from Step 4, or leave empty for prompt-only)
CHARACTER_NAME = ""  #@param {type:"string"}

#@markdown ---
#@markdown **Video Generation Settings**
VIDEO_MODEL = "AnimateDiff (recommended)"  #@param ["AnimateDiff (recommended)", "CogVideoX 1.5"]
NUM_FRAMES = 24  #@param {type:"slider", min:16, max:32, step:4}
ANIM_STEPS = 30  #@param {type:"slider", min:15, max:50, step:5}
ANIM_GUIDANCE = 8.0  #@param {type:"slider", min:3.0, max:15.0, step:0.5}
DENOISING_STRENGTH = 0.40  #@param {type:"slider", min:0.20, max:0.65, step:0.05}
FPS = 8  #@param {type:"slider", min:8, max:16, step:4}
TARGET_FPS = 24  #@param {type:"slider", min:8, max:30, step:4}

#@markdown ---
#@markdown **Post-Processing**
FACE_RESTORE = True  #@param {type:"boolean"}
SPATIAL_UPSCALE = True  #@param {type:"boolean"}
COLOR_GRADE = True  #@param {type:"boolean"}
COLOR_PALETTE = "warm"  #@param ["warm", "cool", "vibrant", "muted"]

#@markdown ---
#@markdown **Image Settings (SDXL keyframes)**
IMAGE_WIDTH = 768  #@param {type:"slider", min:512, max:1024, step:128}
IMAGE_HEIGHT = 1152  #@param {type:"slider", min:512, max:1152, step:128}
SDXL_STEPS = 25  #@param {type:"slider", min:10, max:50, step:5}
SDXL_GUIDANCE = 7.5  #@param {type:"slider", min:1.0, max:15.0, step:0.5}
LORA_SCALE = 1.0  #@param {type:"slider", min:0.5, max:2.0, step:0.1}
NEGATIVE_PROMPT = "blurry, low quality, deformed, extra fingers, worst quality, ugly, disfigured, bad anatomy, bad hands, 3d render, cgi, photorealistic, live action"  #@param {type:"string"}
MOTION_NEGATIVE = "blurry face, distorted mouth, facial blur, fuzzy features, low quality, worst quality, deformed face, blurry hair, camera shake, distorted eyes, asymmetric eyes, 3d render, photorealistic"  #@param {type:"string"}

import os, gc, sys, time, torch, json
import numpy as np
from pathlib import Path
from PIL import Image
import cv2

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])

if GEMINI_API_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

# ================================================================
# Phase 1: Story Decomposition — text → shot script
# ================================================================
print("=" * 60)
print("PHASE 1: Story Decomposition")
print("=" * 60)

from agents.story.decomposer import StoryDecomposer

decomposer = StoryDecomposer(gemini_api_key=GEMINI_API_KEY or None, character_name=CHARACTER_NAME or None)
script = decomposer.decompose(STORY_TEXT)
shots = decomposer.decompose_to_shots(STORY_TEXT)

print(f"\nGenerated script ({len(shots)} shots):")
print("-" * 40)
print(script)
print("-" * 40)

# ================================================================
# Phase 2: SDXL + LoRA — Generate character keyframes
# ================================================================
print("\n" + "=" * 60)
print("PHASE 2: SDXL + LoRA Keyframes")
print("=" * 60)

gc.collect()
torch.cuda.empty_cache()

# Check for trained LoRA
char_id = CHARACTER_NAME.lower().replace(" ", "_") if CHARACTER_NAME else None
lora_dir = WAREHOUSE / "lora" / char_id if char_id else None
has_lora = lora_dir and (lora_dir / "metadata.json").exists() if lora_dir else False

from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

if has_lora:
    from peft import PeftModel
    meta = json.loads((lora_dir / "metadata.json").read_text())
    base_model = meta["base_model"]
    print(f"Loading {base_model} + LoRA for '{CHARACTER_NAME}'…")
else:
    base_model = "cagliostrolab/animagine-xl-3.1"
    print(f"Loading {base_model} (no character LoRA)…")

pipe = StableDiffusionXLPipeline.from_pretrained(base_model, torch_dtype=torch.float16)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

if has_lora:
    pipe.unet = PeftModel.from_pretrained(pipe.unet, str(lora_dir))
    pipe.unet.eval()
    if LORA_SCALE != 1.0:
        for module in pipe.unet.modules():
            if hasattr(module, "scaling"):
                for key in module.scaling:
                    module.scaling[key] = LORA_SCALE
    print(f"LoRA loaded (scale={LORA_SCALE})")

pipe.to("cuda")
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()
try:
    pipe.enable_xformers_memory_efficient_attention()
except Exception:
    pass

keyframes = []
kf_prompts = []
for i, shot in enumerate(shots):
    # Build anime-enhanced prompt
    desc = shot["description"]
    if CHARACTER_NAME and CHARACTER_NAME.lower() not in desc.lower():
        desc = f"{CHARACTER_NAME}, {desc}"
    prompt = f"{desc}, anime style, anime screencap, studio quality, sharp lineart, vibrant colors, masterpiece, best quality"
    kf_prompts.append(prompt)

    gen = torch.Generator("cuda").manual_seed(42 + i)
    result = pipe(
        prompt=prompt,
        negative_prompt=NEGATIVE_PROMPT,
        height=IMAGE_HEIGHT, width=IMAGE_WIDTH,
        num_inference_steps=SDXL_STEPS,
        guidance_scale=SDXL_GUIDANCE,
        generator=gen,
    )
    keyframes.append(result.images[0])
    print(f"  Keyframe {i+1}/{len(shots)} generated: {desc[:60]}…")

# Save keyframes
kf_dir = WAREHOUSE / "outputs" / "text2anime_keyframes"
kf_dir.mkdir(parents=True, exist_ok=True)
for i, kf in enumerate(keyframes):
    kf.save(kf_dir / f"kf_{i:03d}.png")
print(f"Keyframes saved to {kf_dir}")

# Unload SDXL
if has_lora:
    while hasattr(pipe.unet, "base_model"):
        try:
            pipe.unet = pipe.unet.base_model.model
        except Exception:
            break
del pipe
gc.collect()
torch.cuda.empty_cache()
print("SDXL unloaded.\n")

# ================================================================
# Phase 3: Animate Keyframes (AnimateDiff + LoRA or CogVideoX fallback)
# ================================================================
print("=" * 60)
print("PHASE 3: Video Generation")
print("=" * 60)

use_animatediff = VIDEO_MODEL.startswith("AnimateDiff")
all_clips = []
start_time = time.time()

if use_animatediff:
    # ----------------------------------------------------------
    # Phase 3a: Auto-train SD 1.5 LoRA if missing
    # ----------------------------------------------------------
    sd15_lora_dir = None
    if CHARACTER_NAME:
        char_id = CHARACTER_NAME.lower().replace(" ", "_")
        sd15_dir = WAREHOUSE / "lora" / f"{char_id}_sd15"
        sdxl_dir = WAREHOUSE / "lora" / char_id

        if (sd15_dir / "adapter_model.safetensors").exists() or (sd15_dir / "adapter_config.json").exists():
            sd15_lora_dir = sd15_dir
            print(f"SD 1.5 LoRA found at {sd15_dir}")
        elif sdxl_dir.exists():
            # Auto-train SD 1.5 LoRA from existing reference images
            print(f"\nSD 1.5 LoRA not found. Auto-training from reference images...")
            print("(AnimateDiff needs SD 1.5 LoRA for character consistency)")
            try:
                ref_dir = WAREHOUSE / "references" / char_id
                if ref_dir.exists():
                    ref_images = [str(p) for p in ref_dir.glob("*.png")] + [str(p) for p in ref_dir.glob("*.jpg")]
                    if len(ref_images) >= 3:
                        from agents.character.trainer import LoRATrainer
                        trainer = LoRATrainer(str(WAREHOUSE))
                        trainer.train_sd15_lora(
                            character_images=ref_images,
                            character_id=char_id,
                            character_name=CHARACTER_NAME,
                            rank=16,
                            max_steps=300,
                        )
                        if (sd15_dir / "adapter_model.safetensors").exists() or (sd15_dir / "adapter_config.json").exists():
                            sd15_lora_dir = sd15_dir
                            print(f"SD 1.5 LoRA auto-trained successfully!")
                        else:
                            print("SD 1.5 LoRA training completed but files not found. Continuing without LoRA.")
                    else:
                        print(f"  Only {len(ref_images)} reference images found (need >= 3). Skipping SD 1.5 LoRA training.")
                else:
                    print(f"  No reference images at {ref_dir}. Skipping SD 1.5 LoRA training.")
            except Exception as e_train:
                print(f"  SD 1.5 LoRA auto-training failed: {e_train}")
                print("  Continuing without character LoRA (prompt-only mode)")
            gc.collect()
            torch.cuda.empty_cache()

    # ----------------------------------------------------------
    # Phase 3b: Load AnimateDiff pipeline with anime base model
    # ----------------------------------------------------------
    print("\nLoading AnimateDiff + anime base model...")
    from diffusers import AnimateDiffPipeline, MotionAdapter, DDIMScheduler

    motion_adapter = MotionAdapter.from_pretrained(
        "guoyww/animatediff-motion-adapter-v1-5-3",
        torch_dtype=torch.float16,
        cache_dir=str(WAREHOUSE / "models"),
    )
    print("  Motion adapter loaded")

    # Try anime base models in order
    _SD15_MODELS = ["Linaqruf/anything-v3-1", "Lykon/dreamshaper-8", "runwayml/stable-diffusion-v1-5"]
    anim_pipe = None
    for _model_id in _SD15_MODELS:
        try:
            anim_pipe = AnimateDiffPipeline.from_pretrained(
                _model_id,
                motion_adapter=motion_adapter,
                torch_dtype=torch.float16,
                cache_dir=str(WAREHOUSE / "models"),
            )
            print(f"  Base model loaded: {_model_id}")
            break
        except Exception as _e:
            print(f"  {_model_id} failed: {_e}")
            continue

    if anim_pipe is None:
        print("ERROR: No SD 1.5 base model available. Falling back to CogVideoX.")
        use_animatediff = False
    else:
        anim_pipe.scheduler = DDIMScheduler.from_config(
            anim_pipe.scheduler.config,
            beta_schedule="linear",
            clip_sample=False,
        )
        anim_pipe.enable_vae_slicing()
        anim_pipe.enable_model_cpu_offload()

        # Load SD 1.5 LoRA for character consistency
        _lora_loaded = False
        if sd15_lora_dir is not None:
            try:
                from peft import PeftModel
                anim_pipe.unet = PeftModel.from_pretrained(anim_pipe.unet, str(sd15_lora_dir))
                anim_pipe.unet.eval()
                _lora_loaded = True
                print(f"  Character LoRA loaded from {sd15_lora_dir}")
            except Exception as e_lora:
                print(f"  LoRA loading failed: {e_lora}. Continuing with prompt-only mode.")

        print("AnimateDiff pipeline ready!\n")

        # ----------------------------------------------------------
        # Phase 3c: Generate clips (AnimateDiff vid2vid from keyframes)
        # ----------------------------------------------------------
        ANIM_NEGATIVE = "low quality, bad anatomy, worst quality, blurry, deformed, disfigured, static, ugly, watermark, text, extra limbs"

        for i, kf_image in enumerate(keyframes):
            shot = shots[i]
            motion_prompt = f"1girl, anime screencap, {shot['description']}, clear detailed face, expressive eyes, smooth fluid motion, anime style, high quality animation, masterpiece"
            if CHARACTER_NAME:
                motion_prompt = f"{CHARACTER_NAME}, " + motion_prompt
            print(f"  Animating shot {i+1}/{len(keyframes)}: \"{shot['description'][:50]}...\"")

            # Resize keyframe for AnimateDiff (SD 1.5: 512x768 portrait)
            kf_resized = kf_image.resize((512, 768), Image.LANCZOS)

            gen = torch.Generator("cpu").manual_seed(42 + i)
            try:
                result = anim_pipe(
                    prompt=motion_prompt,
                    negative_prompt=ANIM_NEGATIVE,
                    num_frames=NUM_FRAMES,
                    width=512,
                    height=768,
                    num_inference_steps=ANIM_STEPS,
                    guidance_scale=ANIM_GUIDANCE,
                    generator=gen,
                )
                clip_frames = result.frames[0]

                # Blend keyframe into first few frames for identity anchoring
                if len(clip_frames) > 3:
                    try:
                        clip_w, clip_h = clip_frames[0].size if hasattr(clip_frames[0], 'size') else (np.array(clip_frames[0]).shape[1], np.array(clip_frames[0]).shape[0])
                        kf_for_blend = kf_resized.resize((clip_w, clip_h), Image.LANCZOS)
                        for blend_j in range(min(3, len(clip_frames))):
                            alpha = 0.3 + (blend_j / 3.0) * 0.7  # start at 30% keyframe, increase to video
                            frame_pil = clip_frames[blend_j] if isinstance(clip_frames[blend_j], Image.Image) else Image.fromarray(np.array(clip_frames[blend_j]))
                            clip_frames[blend_j] = Image.blend(kf_for_blend, frame_pil, alpha)
                    except Exception as e_blend:
                        pass  # blend is optional

                all_clips.append(clip_frames)
                print(f"  Shot {i+1} done \u2014 {len(clip_frames)} frames")

            except Exception as e_shot:
                print(f"  Shot {i+1} AnimateDiff failed: {e_shot}")
                # Create a static clip from keyframe as fallback
                all_clips.append([kf_resized] * 16)
                print(f"  Shot {i+1} using static keyframe fallback (16 frames)")

            gc.collect()
            torch.cuda.empty_cache()

        # Cleanup AnimateDiff
        anim_time = time.time() - start_time
        print(f"\nAnimateDiff done in {anim_time/60:.1f} minutes.")
        # Unwrap PEFT before deleting
        if _lora_loaded:
            while hasattr(anim_pipe.unet, "base_model"):
                try:
                    anim_pipe.unet = anim_pipe.unet.base_model.model
                except Exception:
                    break
        del anim_pipe, motion_adapter
        gc.collect()
        torch.cuda.empty_cache()

# ----------------------------------------------------------
# CogVideoX fallback (if AnimateDiff not selected or failed)
# ----------------------------------------------------------
if not use_animatediff or len(all_clips) == 0:
    print("\nUsing CogVideoX 1.5 fallback...")
    from diffusers import CogVideoXImageToVideoPipeline
    try:
        from optimum.quanto import qint8, quantize, freeze
        cogvid_pipe = CogVideoXImageToVideoPipeline.from_pretrained(
            "THUDM/CogVideoX1.5-5B-I2V",
            torch_dtype=torch.bfloat16,
        )
        quantize(cogvid_pipe.transformer, weights=qint8)
        freeze(cogvid_pipe.transformer)
        quantize(cogvid_pipe.text_encoder, weights=qint8)
        freeze(cogvid_pipe.text_encoder)
        cogvid_pipe.enable_model_cpu_offload()
        cogvid_pipe.vae.enable_tiling()
        cogvid_pipe.vae.enable_slicing()
        print("CogVideoX 1.5 loaded.")

        all_clips = []
        for i, kf_image in enumerate(keyframes):
            shot = shots[i]
            motion_prompt = f"Slow tracking shot of anime character, clear detailed face, expressive eyes, sharp features, {shot['description']}, smooth fluid motion, gradually moving, anime style, high quality animation"
            print(f"\n  Animating shot {i+1}/{len(keyframes)}: \"{shot['description'][:50]}...\"")

            kf_w, kf_h = kf_image.size
            target_ratio = 480 / 720
            current_ratio = kf_w / kf_h
            if current_ratio > target_ratio:
                new_w = int(kf_h * target_ratio)
                left = (kf_w - new_w) // 2
                kf_cropped = kf_image.crop((left, 0, left + new_w, kf_h))
            else:
                new_h = int(kf_w / target_ratio)
                top = (kf_h - new_h) // 2
                kf_cropped = kf_image.crop((0, top, kf_w, top + new_h))
            kf_resized = kf_cropped.resize((480, 720), Image.LANCZOS)

            gen = torch.Generator("cpu").manual_seed(42 + i)
            output = cogvid_pipe(
                image=kf_resized,
                prompt=motion_prompt,
                negative_prompt=MOTION_NEGATIVE,
                num_frames=49,
                num_inference_steps=60,
                guidance_scale=6.0,
                use_dynamic_cfg=True,
                generator=gen,
            )
            clip_frames = output.frames[0]
            all_clips.append(clip_frames)
            elapsed = time.time() - start_time
            print(f"  Shot {i+1} done \u2014 {len(clip_frames)} frames ({elapsed:.0f}s elapsed)")
            gc.collect()
            torch.cuda.empty_cache()

        del cogvid_pipe
        gc.collect()
        torch.cuda.empty_cache()
    except Exception as e_cog:
        print(f"CogVideoX also failed: {e_cog}")
        print("Using static keyframes as fallback.")
        all_clips = [[kf.resize((512, 768), Image.LANCZOS)] * 16 for kf in keyframes]

# ================================================================
# Phase 3b: Trim Static Tails (fixes frozen ending)
# ================================================================
print("\nTrimming static tail frames...")
try:
    from agents.postprocess.motion_trim import MotionTrimmer
    trimmer = MotionTrimmer()
    for trim_idx, clip in enumerate(all_clips):
        original_len = len(clip)
        all_clips[trim_idx] = trimmer.trim_static_tail(clip)
        trimmed = original_len - len(all_clips[trim_idx])
        if trimmed > 0:
            print(f"  Clip {trim_idx+1}: {original_len} \u2192 {len(all_clips[trim_idx])} frames (trimmed {trimmed} static frames)")
        else:
            print(f"  Clip {trim_idx+1}: {original_len} frames (no static tail detected)")
    # Extend last clip with ping-pong to avoid abrupt ending
    if len(all_clips) > 0 and len(all_clips[-1]) < 35:
        all_clips[-1] = trimmer.extend_last_clip_with_pingpong(all_clips[-1], target_frames=40)
        print(f"  Last clip extended to {len(all_clips[-1])} frames via ping-pong")
except Exception as e:
    print(f"  Motion trimming skipped ({e})")

# ================================================================
# Phase 4: Post-Processing — face restore + sharpen + color grade
# ================================================================
print("\n" + "=" * 60)
print("PHASE 4: Post-Processing")
print("=" * 60)

if FACE_RESTORE:
    # Anime face restoration (replaces GFPGAN which doesn’t work on anime faces)
    try:
        from agents.postprocess.face_restore import AnimeFaceRestorer
        restorer = AnimeFaceRestorer()
        print("Restoring anime faces…")
        for clip_idx, clip in enumerate(all_clips):
            all_clips[clip_idx] = restorer.restore_frames(clip)
        print("Anime face restoration done.")
    except Exception as e:
        print(f"Face restoration skipped ({e})")

if SPATIAL_UPSCALE:
    # Ensure torchvision compatibility shim is active for basicsr
    import torchvision as _tv
    if not hasattr(_tv.transforms, 'functional_tensor'):
        import torchvision.transforms.functional as _F2
        import types as _types2
        _ftmod2 = _types2.ModuleType('torchvision.transforms.functional_tensor')
        for _a in ['rgb_to_grayscale', 'normalize', 'resize', 'pad']:
            if hasattr(_F2, _a):
                setattr(_ftmod2, _a, getattr(_F2, _a))
        sys.modules['torchvision.transforms.functional_tensor'] = _ftmod2
        _tv.transforms.functional_tensor = _ftmod2

if SPATIAL_UPSCALE:
    # Frame sharpening with Real-ESRGAN anime
    try:
        from basicsr.archs.rrdbnet_arch import RRDBNet
        from realesrgan import RealESRGANer
        esrgan_model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=6, num_grow_ch=32, scale=4)
        esrgan = RealESRGANer(
            scale=4,
            model_path="https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth",
            model=esrgan_model, half=True,
        )
        print("Sharpening frames with Real-ESRGAN anime…")
        for clip_idx, clip in enumerate(all_clips):
            sharpened = []
            for frame_pil in clip:
                frame_bgr = cv2.cvtColor(np.array(frame_pil), cv2.COLOR_RGB2BGR)
                h_orig, w_orig = frame_bgr.shape[:2]
                out_bgr, _ = esrgan.enhance(frame_bgr, outscale=1)
                if out_bgr.shape[:2] != (h_orig, w_orig):
                    out_bgr = cv2.resize(out_bgr, (w_orig, h_orig), interpolation=cv2.INTER_LANCZOS4)
                sharpened.append(Image.fromarray(cv2.cvtColor(out_bgr, cv2.COLOR_BGR2RGB)))
            all_clips[clip_idx] = sharpened
        del esrgan, esrgan_model
        gc.collect()
        torch.cuda.empty_cache()
        print("Frame sharpening done.")
    except Exception as e:
        print(f"Real-ESRGAN sharpening skipped ({e})")

if COLOR_GRADE:
    # Anime color grading
    from agents.postprocess.color_grade import AnimeColorGrader
    grader = AnimeColorGrader()
    print(f"Applying '{COLOR_PALETTE}' color grading…")
    for clip_idx, clip in enumerate(all_clips):
        all_clips[clip_idx] = grader.grade_with_palette(clip, palette=COLOR_PALETTE)
    print("Color grading done.")

# ================================================================
# Phase 5: Temporal Upscaling (RIFE) — 8fps → 24fps
# ================================================================
if TARGET_FPS > FPS:
    print(f"\nTemporal upscaling: {FPS}fps → {TARGET_FPS}fps…")
    multiplier = round(TARGET_FPS / FPS)
    # Try RIFE for proper optical-flow interpolation (no ghosting)
    rife_available = False
    try:
        from agents.postprocess.upscaler import VideoUpscaler
        _upscaler = VideoUpscaler(str(WAREHOUSE))
        _rife_model = _upscaler._load_rife()
        if _rife_model is not None:
            rife_available = True
            print("  Using RIFE for temporal upscaling (optical flow)")
    except Exception:
        pass

    for clip_idx, clip in enumerate(all_clips):
        original_len = len(clip)
        if rife_available:
            try:
                all_clips[clip_idx] = _upscaler._rife_interpolate_sequence(_rife_model, clip, multiplier)
            except Exception as e_rife:
                print(f"  RIFE failed for clip {clip_idx+1}: {e_rife}, using linear blend")
                upscaled = []
                for j in range(len(clip) - 1):
                    upscaled.append(clip[j])
                    a1 = np.array(clip[j]).astype(np.float32)
                    a2 = np.array(clip[j + 1]).astype(np.float32)
                    for k in range(1, multiplier):
                        alpha = k / multiplier
                        blended = ((1 - alpha) * a1 + alpha * a2).astype(np.uint8)
                        upscaled.append(Image.fromarray(blended))
                upscaled.append(clip[-1])
                all_clips[clip_idx] = upscaled
        else:
            upscaled = []
            for j in range(len(clip) - 1):
                upscaled.append(clip[j])
                a1 = np.array(clip[j]).astype(np.float32)
                a2 = np.array(clip[j + 1]).astype(np.float32)
                for k in range(1, multiplier):
                    alpha = k / multiplier
                    blended = ((1 - alpha) * a1 + alpha * a2).astype(np.uint8)
                    upscaled.append(Image.fromarray(blended))
            upscaled.append(clip[-1])
            all_clips[clip_idx] = upscaled
        print(f"  Clip {clip_idx+1}: {original_len} → {len(all_clips[clip_idx])} frames")

    # Clean up RIFE to free VRAM
    if rife_available:
        try:
            _upscaler._unload_rife()
        except Exception:
            pass
        gc.collect()
        torch.cuda.empty_cache()

    output_fps = TARGET_FPS
else:
    output_fps = FPS

# ================================================================
# Phase 6: Assembly — cross-dissolve stitch
# ================================================================
print(f"\nAssembling {len(all_clips)} clips with cross-dissolve transitions…")
CROSSFADE_FRAMES = 6

all_frames = []
for clip_idx, clip in enumerate(all_clips):
    clip_arrays = [np.array(f) for f in clip]
    if clip_idx == 0:
        all_frames.extend(clip_arrays)
    else:
        overlap = min(CROSSFADE_FRAMES, len(all_frames), len(clip_arrays))
        for j in range(overlap):
            t = (j + 1) / (overlap + 1)
            prev_f = all_frames[-(overlap - j)].astype(np.float32)
            next_f = clip_arrays[j].astype(np.float32)
            blended = ((1 - t) * prev_f + t * next_f).clip(0, 255).astype(np.uint8)
            all_frames[-(overlap - j)] = blended
        all_frames.extend(clip_arrays[overlap:])

duration = len(all_frames) / output_fps
print(f"Total: {len(all_frames)} frames ({duration:.1f}s at {output_fps}fps)")

# Save video
output_dir = WAREHOUSE / "outputs" / "text2anime"
output_dir.mkdir(parents=True, exist_ok=True)
video_path = output_dir / f"text2anime_{int(time.time())}.mp4"

h, w = all_frames[0].shape[:2]
writer = cv2.VideoWriter(str(video_path), cv2.VideoWriter_fourcc(*"mp4v"), output_fps, (w, h))
for frame in all_frames:
    writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
writer.release()

size_mb = video_path.stat().st_size / 1e6
total_time = time.time() - start_time

print(f"\n{'=' * 60}")
print(f"TEXT-TO-ANIME COMPLETE!")
print(f"{'=' * 60}")
print(f"Video: {video_path}")
print(f"Duration: {duration:.1f}s | Resolution: {w}x{h} | FPS: {output_fps}")
print(f"Shots: {len(shots)} | Size: {size_mb:.1f}MB")
print(f"Total time: {total_time/60:.1f} minutes")
print(f"Quality: {QUALITY} | Color: {COLOR_PALETTE}")

# Show keyframe preview
import matplotlib.pyplot as plt
cols = min(len(keyframes), 4)
rows = (len(keyframes) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
axes_flat = np.array(axes).flat if hasattr(axes, 'flat') else [axes]
for i, (ax, kf) in enumerate(zip(axes_flat, keyframes)):
    ax.imshow(kf)
    shot_desc = shots[i]["description"][:40] if i < len(shots) else ""
    ax.set_title(f"Shot {i+1}: {shot_desc}…", fontsize=9)
    ax.axis("off")
for j in range(i + 1, rows * cols):
    try:
        axes_flat[j].axis("off")
    except Exception:
        pass
plt.suptitle("Text-to-Anime — Generated Keyframes", fontsize=13)
plt.tight_layout()
plt.show()

gc.collect()
torch.cuda.empty_cache()
print("\nDone. VRAM freed.")

---

## Quick Reference

| Step | What | Time (Free/Pro) |
|------|------|-----------------|
| 1 | Setup | 2 min |
| 2 | Get images | 1–5 min |
| 3 | Auto-caption (BLIP) | 2–5 min |
| 4 | Train LoRA (SDXL) | 15–40 min |
| 4b | Batch train | N × step 4 |
| 4c | Train SD 1.5 LoRA | 10–20 min |
| 5 | Test character | 2 min |
| 6 | Register in memory | 1 min |
| 7 | Dashboard + export | 1 min |
| 8 | Colab survival | Background |
| 9 | Single anime clip | 10–30 min (A100) |
| 10 | 2-minute video | 60–120 min (A100) |
| **10b** | **Text-to-Anime (NEW)** | **15–60 min (A100)** |
| 11 | Interactive Studio | Ongoing |

**Text-to-Anime (Step 10b)** is the Sora-like feature — just type a story and get anime video with:
- Story decomposition (Gemini Flash or local NLP)
- Character consistency (SDXL + LoRA keyframes)
- Real motion (CogVideoX 1.5)
- Post-processing (GFPGAN + Real-ESRGAN + color grading)
- Smooth 24fps output with cross-dissolve transitions

## 11. AnimeLoom Interactive Studio (Gradio UI)
Launch a visual interface to configure settings, **preview a test keyframe** before committing, and generate video — all from one panel. Run this cell instead of Steps 9/10 for a more interactive workflow.

In [ ]:
#@title 11. AnimeLoom Interactive Studio
#@markdown Launch the Gradio UI for interactive anime video generation.

import os, gc, sys, time, torch, json, subprocess
import numpy as np
from pathlib import Path
from PIL import Image
import gradio as gr

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])

# --- Discover trained characters ---
def get_trained_characters():
    lora_dir = WAREHOUSE / "lora"
    chars = []
    if lora_dir.exists():
        for d in sorted(lora_dir.iterdir()):
            if (d / "metadata.json").exists():
                chars.append(d.name)
    return chars if chars else ["(none trained)"]

# --- Preview: generate 1 test keyframe + stats ---
def preview_keyframe(char_name, width, height, steps, guidance, lora_scale,
                     kf_prompts_text, neg_prompt, cogvid_steps, cogvid_guidance,
                     num_frames, fps, face_restore):
    char_id = char_name.lower().replace(" ", "_")
    lora_dir = WAREHOUSE / "lora" / char_id
    meta_path = lora_dir / "metadata.json"

    if not meta_path.exists():
        return None, f"No LoRA found for '{char_name}'. Run Step 4 first."

    kf_prompts = [p.strip() for p in kf_prompts_text.strip().split("\n") if p.strip()]
    num_kf = len(kf_prompts)
    total_frames = num_kf * num_frames
    crossfade = 4
    stitched_frames = total_frames - (num_kf - 1) * crossfade
    duration = stitched_frames / fps

    # Stats text
    stats = f"""**Preview Stats**
- Character: {char_name}
- SDXL resolution: {width}×{height} (portrait)
- CogVideoX input: 480×720
- Keyframes: {num_kf}
- CogVideoX frames per clip: {num_frames}
- Total frames after stitch: ~{stitched_frames}
- Estimated duration: {duration:.1f}s @ {fps}fps
- CogVideoX steps: {cogvid_steps} | Guidance: {cogvid_guidance}
- Face restore: {'Yes (GFPGAN + Real-ESRGAN)' if face_restore else 'No'}
- Estimated generation time: ~{num_kf * 3:.0f} min (A100)
"""

    # Generate 1 test keyframe
    gc.collect()
    torch.cuda.empty_cache()

    meta = json.loads(meta_path.read_text())
    base_model = meta["base_model"]

    from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
    from peft import PeftModel

    pipe = StableDiffusionXLPipeline.from_pretrained(base_model, torch_dtype=torch.float16)
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    pipe.unet = PeftModel.from_pretrained(pipe.unet, str(lora_dir))
    pipe.unet.eval()

    if lora_scale != 1.0:
        for module in pipe.unet.modules():
            if hasattr(module, "scaling"):
                for key in module.scaling:
                    module.scaling[key] = lora_scale

    pipe.to("cuda")
    pipe.enable_vae_slicing()
    pipe.enable_vae_tiling()
    try:
        pipe.enable_xformers_memory_efficient_attention()
    except Exception:
        pass

    gen = torch.Generator("cuda").manual_seed(42)
    result = pipe(
        prompt=kf_prompts[0],
        negative_prompt=neg_prompt,
        height=int(height), width=int(width),
        num_inference_steps=int(steps),
        guidance_scale=guidance,
        generator=gen,
    )
    preview_img = result.images[0]

    # Save preview
    preview_path = WAREHOUSE / "outputs" / f"preview_{char_id}.png"
    preview_img.save(preview_path)

    # Unload
    while hasattr(pipe.unet, "base_model"):
        try:
            pipe.unet = pipe.unet.base_model.model
        except Exception:
            break
    del pipe
    gc.collect()
    torch.cuda.empty_cache()

    return preview_img, stats

# --- Full video generation ---
def generate_video(char_name, width, height, sdxl_steps, sdxl_guidance, lora_scale,
                   kf_prompts_text, motion_prompts_text, neg_prompt, motion_neg,
                   cogvid_steps, cogvid_guidance, num_frames, fps, face_restore,
                   denoising_strength, progress=gr.Progress()):
    import cv2
    char_id = char_name.lower().replace(" ", "_")
    lora_dir = WAREHOUSE / "lora" / char_id
    meta_path = lora_dir / "metadata.json"

    if not meta_path.exists():
        return None, None, f"No LoRA found for '{char_name}'."

    kf_prompts = [p.strip() for p in kf_prompts_text.strip().split("\n") if p.strip()]
    motion_prompts = [p.strip() for p in motion_prompts_text.strip().split("\n") if p.strip()]
    num_kf = len(kf_prompts)

    meta = json.loads(meta_path.read_text())
    base_model = meta["base_model"]

    # Phase 1: SDXL keyframes
    progress(0.0, desc="Loading SDXL + LoRA…")
    gc.collect()
    torch.cuda.empty_cache()

    from diffusers import StableDiffusionXLPipeline, AutoPipelineForImage2Image, DPMSolverMultistepScheduler
    from peft import PeftModel

    pipe = StableDiffusionXLPipeline.from_pretrained(base_model, torch_dtype=torch.float16)
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    pipe.unet = PeftModel.from_pretrained(pipe.unet, str(lora_dir))
    pipe.unet.eval()

    if lora_scale != 1.0:
        for module in pipe.unet.modules():
            if hasattr(module, "scaling"):
                for key in module.scaling:
                    module.scaling[key] = lora_scale

    pipe.to("cuda")
    pipe.enable_vae_slicing()
    pipe.enable_vae_tiling()
    try:
        pipe.enable_xformers_memory_efficient_attention()
    except Exception:
        pass

    pipe_i2i = AutoPipelineForImage2Image.from_pipe(pipe)

    keyframes = []
    for i, prompt in enumerate(kf_prompts):
        progress((i + 1) / (num_kf * 3), desc=f"Keyframe {i+1}/{num_kf}…")
        gen = torch.Generator("cuda").manual_seed(42 + i)
        if i == 0:
            result = pipe(prompt=prompt, negative_prompt=neg_prompt,
                          height=int(height), width=int(width),
                          num_inference_steps=int(sdxl_steps), guidance_scale=sdxl_guidance, generator=gen)
        else:
            result = pipe_i2i(prompt=prompt, negative_prompt=neg_prompt,
                              image=keyframes[-1], strength=denoising_strength,
                              num_inference_steps=int(sdxl_steps), guidance_scale=sdxl_guidance, generator=gen)
        keyframes.append(result.images[0])

    # Save keyframes
    kf_dir = WAREHOUSE / "outputs" / f"keyframes_{char_id}"
    kf_dir.mkdir(parents=True, exist_ok=True)
    for i, kf in enumerate(keyframes):
        kf.save(kf_dir / f"kf_{i:03d}.png")

    while hasattr(pipe.unet, "base_model"):
        try:
            pipe.unet = pipe.unet.base_model.model
        except Exception:
            break
    del pipe, pipe_i2i
    gc.collect()
    torch.cuda.empty_cache()

    # Phase 2: CogVideoX animation
    progress(num_kf / (num_kf * 3), desc="Loading CogVideoX 1.5…")

    from diffusers import CogVideoXImageToVideoPipeline
    from optimum.quanto import qint8, quantize, freeze

    cogvid_pipe = CogVideoXImageToVideoPipeline.from_pretrained(
        "THUDM/CogVideoX1.5-5B-I2V", torch_dtype=torch.bfloat16)
    quantize(cogvid_pipe.transformer, weights=qint8)
    freeze(cogvid_pipe.transformer)
    quantize(cogvid_pipe.text_encoder, weights=qint8)
    freeze(cogvid_pipe.text_encoder)
    cogvid_pipe.enable_model_cpu_offload()
    cogvid_pipe.vae.enable_tiling()
    cogvid_pipe.vae.enable_slicing()

    all_clips = []
    for i, kf_image in enumerate(keyframes):
        motion_prompt = motion_prompts[i % len(motion_prompts)]
        progress((num_kf + i + 1) / (num_kf * 3), desc=f"Animating clip {i+1}/{num_kf}…")

        kf_w, kf_h = kf_image.size
        target_ratio = 480 / 720
        current_ratio = kf_w / kf_h
        if current_ratio > target_ratio:
            new_w = int(kf_h * target_ratio)
            left = (kf_w - new_w) // 2
            kf_cropped = kf_image.crop((left, 0, left + new_w, kf_h))
        else:
            new_h = int(kf_w / target_ratio)
            top = (kf_h - new_h) // 2
            kf_cropped = kf_image.crop((0, top, kf_w, top + new_h))
        kf_resized = kf_cropped.resize((480, 720), Image.LANCZOS)

        gen = torch.Generator("cpu").manual_seed(42 + i)
        output = cogvid_pipe(image=kf_resized, prompt=motion_prompt, negative_prompt=motion_neg,
                             num_frames=int(num_frames), num_inference_steps=int(cogvid_steps),
                             guidance_scale=cogvid_guidance, use_dynamic_cfg=True, generator=gen)
        all_clips.append(output.frames[0])
        gc.collect()
        torch.cuda.empty_cache()

    del cogvid_pipe
    gc.collect()
    torch.cuda.empty_cache()

    # Phase 3: Face restoration + sharpening
    if face_restore:
        progress(2 * num_kf / (num_kf * 3), desc="Restoring faces…")
        try:
            from gfpgan import GFPGANer
            restorer = GFPGANer(
                model_path="https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth",
                upscale=1, arch="clean", channel_multiplier=2, bg_upsampler=None)
            for clip_idx, clip in enumerate(all_clips):
                restored = []
                for frame_pil in clip:
                    frame_bgr = cv2.cvtColor(np.array(frame_pil), cv2.COLOR_RGB2BGR)
                    _, _, out_bgr = restorer.enhance(frame_bgr, has_aligned=False, only_center_face=False, paste_back=True)
                    restored.append(Image.fromarray(cv2.cvtColor(out_bgr, cv2.COLOR_BGR2RGB)))
                all_clips[clip_idx] = restored
            del restorer
            gc.collect()
            torch.cuda.empty_cache()
        except Exception:
            pass

        progress(2.3 * num_kf / (num_kf * 3), desc="Sharpening frames…")
        try:
            from basicsr.archs.rrdbnet_arch import RRDBNet
            from realesrgan import RealESRGANer
            esrgan_model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=6, num_grow_ch=32, scale=4)
            esrgan = RealESRGANer(
                scale=4, model_path="https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth",
                dni_weight=None, model=esrgan_model, half=True)
            for clip_idx, clip in enumerate(all_clips):
                sharpened = []
                for frame_pil in clip:
                    frame_bgr = cv2.cvtColor(np.array(frame_pil), cv2.COLOR_RGB2BGR)
                    h_orig, w_orig = frame_bgr.shape[:2]
                    out_bgr, _ = esrgan.enhance(frame_bgr, outscale=1)
                    if out_bgr.shape[:2] != (h_orig, w_orig):
                        out_bgr = cv2.resize(out_bgr, (w_orig, h_orig), interpolation=cv2.INTER_LANCZOS4)
                    sharpened.append(Image.fromarray(cv2.cvtColor(out_bgr, cv2.COLOR_BGR2RGB)))
                all_clips[clip_idx] = sharpened
            del esrgan, esrgan_model
            gc.collect()
            torch.cuda.empty_cache()
        except Exception:
            pass

    # Phase 4: Stitch + save
    progress(0.9, desc="Stitching video…")
    CROSSFADE = 4
    all_frames = []
    for clip_idx, clip in enumerate(all_clips):
        clip_arrays = [np.array(f) for f in clip]
        if clip_idx == 0:
            all_frames.extend(clip_arrays)
        else:
            overlap = min(CROSSFADE, len(all_frames), len(clip_arrays))
            for j in range(overlap):
                t = (j + 1) / (overlap + 1)
                prev = all_frames[-(overlap - j)].astype(np.float32)
                nxt = clip_arrays[j].astype(np.float32)
                all_frames[-(overlap - j)] = ((1 - t) * prev + t * nxt).clip(0, 255).astype(np.uint8)
            all_frames.extend(clip_arrays[overlap:])

    output_dir = WAREHOUSE / "outputs" / f"cogvid_{char_id}"
    output_dir.mkdir(parents=True, exist_ok=True)
    video_path = output_dir / f"{char_id}_studio.mp4"

    h, w = all_frames[0].shape[:2]
    writer = cv2.VideoWriter(str(video_path), cv2.VideoWriter_fourcc(*"mp4v"), int(fps), (w, h))
    for frame in all_frames:
        writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
    writer.release()

    duration = len(all_frames) / fps
    size_mb = video_path.stat().st_size / 1e6

    gc.collect()
    torch.cuda.empty_cache()
    progress(1.0, desc="Done!")

    status = f"**Video saved!** {duration:.1f}s | {len(all_frames)} frames | {size_mb:.1f}MB"
    return keyframes, str(video_path), status

# ================================================================
# Build Gradio UI
# ================================================================
chars = get_trained_characters()

with gr.Blocks(title="AnimeLoom Studio", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# AnimeLoom Interactive Studio\nConfigure, preview, then generate anime video.")

    with gr.Row():
        # --- Left: Settings ---
        with gr.Column(scale=1):
            gr.Markdown("### Settings")
            char_dd = gr.Dropdown(choices=chars, value=chars[0], label="Character")
            refresh_btn = gr.Button("Refresh Characters", size="sm")

            with gr.Accordion("SDXL Settings", open=True):
                width_sl = gr.Slider(512, 1024, value=512, step=128, label="Width")
                height_sl = gr.Slider(512, 1024, value=768, step=128, label="Height")
                sdxl_steps_sl = gr.Slider(10, 50, value=30, step=5, label="SDXL Steps")
                sdxl_guide_sl = gr.Slider(1.0, 15.0, value=7.5, step=0.5, label="SDXL Guidance")
                lora_sl = gr.Slider(0.5, 2.0, value=1.0, step=0.1, label="LoRA Scale")
                denoise_sl = gr.Slider(0.3, 0.8, value=0.45, step=0.05, label="Denoising Strength")

            with gr.Accordion("CogVideoX Settings", open=True):
                cogvid_steps_sl = gr.Slider(20, 100, value=60, step=5, label="CogVideoX Steps")
                cogvid_guide_sl = gr.Slider(1.0, 12.0, value=7.5, step=0.5, label="CogVideoX Guidance")
                num_frames_sl = gr.Slider(13, 49, value=49, step=12, label="Frames per Clip")
                fps_sl = gr.Slider(8, 24, value=16, step=4, label="FPS")

            face_cb = gr.Checkbox(value=True, label="Face Restore (GFPGAN + Real-ESRGAN)")

            with gr.Accordion("Prompts", open=False):
                neg_tb = gr.Textbox(
                    value="blurry, low quality, deformed, extra fingers, worst quality, ugly, disfigured, bad anatomy, bad hands, 3d render, cgi, photorealistic, live action",
                    label="Negative Prompt", lines=2)
                kf_tb = gr.Textbox(
                    value="denji, 1boy, full body, head to toe, facing viewer, front view, detailed face, detailed hair strands, symmetrical eyes, standing, looking at viewer, blonde messy hair, sharp teeth, school uniform, anime screencap, studio quality, sharp lineart, vibrant colors, masterpiece, best quality\ndenji, 1boy, full body, head to toe, facing viewer, front view, detailed face, detailed hair strands, symmetrical eyes, standing, slight grin, blonde messy hair, sharp teeth, school uniform, anime screencap, studio quality, sharp lineart, vibrant colors, masterpiece, best quality\ndenji, 1boy, full body, head to toe, facing viewer, front view, detailed face, detailed hair strands, symmetrical eyes, standing, looking to the side, blonde messy hair, school uniform, anime screencap, studio quality, sharp lineart, vibrant colors, masterpiece, best quality\ndenji, 1boy, full body, head to toe, facing viewer, front view, detailed face, detailed hair strands, symmetrical eyes, standing, serious expression, blonde messy hair, school uniform, anime screencap, studio quality, sharp lineart, vibrant colors, masterpiece, best quality\ndenji, 1boy, full body, head to toe, facing viewer, front view, detailed face, detailed hair strands, symmetrical eyes, standing, relaxed pose, blonde messy hair, school uniform, anime screencap, studio quality, sharp lineart, vibrant colors, masterpiece, best quality\ndenji, 1boy, full body, head to toe, facing viewer, front view, detailed face, detailed hair strands, symmetrical eyes, standing, confident smirk, blonde messy hair, sharp teeth, school uniform, anime screencap, studio quality, sharp lineart, vibrant colors, masterpiece, best quality",
                    label="Keyframe Prompts (one per line)", lines=6)
                motion_tb = gr.Textbox(
                    value="anime boy standing, full body visible, subtle weight shift, slow deliberate blink, hair gently swaying each strand clearly defined, warm lighting, detailed face, sharp facial features, clear expression, symmetrical eyes, stable eye shape, clear detailed eyes throughout, consistent pose, stable camera angle\nanime boy standing, full body visible, slight grin, soft head tilt, calm atmosphere, detailed face, sharp facial features, clear expression, symmetrical eyes, stable eye shape, clear detailed eyes throughout, consistent pose, stable camera angle\nanime boy standing, full body visible, glancing to the side, hair gently swaying each strand clearly defined, serene expression, detailed face, sharp facial features, clear expression, symmetrical eyes, stable eye shape, clear detailed eyes throughout, consistent pose, stable camera angle\nanime boy standing, full body visible, serious focused expression, slight eye movement, dramatic lighting, detailed face, sharp facial features, clear expression, symmetrical eyes, stable eye shape, clear detailed eyes throughout, consistent pose, stable camera angle\nanime boy standing, full body visible, relaxed posture, subtle breathing, soft light, detailed face, sharp facial features, clear expression, symmetrical eyes, stable eye shape, consistent pose, stable camera angle\nanime boy standing, full body visible, confident smirk, looking directly at viewer, determined expression, detailed face, sharp facial features, clear expression, symmetrical eyes, stable eye shape, clear detailed eyes throughout, consistent pose, stable camera angle",
                    label="Motion Prompts (one per line)", lines=6)
                motion_neg_tb = gr.Textbox(
                    value="blurry face, distorted mouth, mouth blur, facial blur, inconsistent facial expression, fuzzy features, low quality, worst quality, deformed face, blurry hair, undefined hair strands, camera shake, shifting perspective, rotating view, distorted eyes, asymmetric eyes, warped eyes, misaligned pupils, blurry eyes, unfocused eyes, 3d render, photorealistic",
                    label="Motion Negative Prompt", lines=2)

        # --- Right: Preview + Output ---
        with gr.Column(scale=2):
            gr.Markdown("### Preview & Output")

            with gr.Row():
                preview_btn = gr.Button("Preview Keyframe (~10s)", variant="secondary", size="lg")
                generate_btn = gr.Button("Generate Full Video", variant="primary", size="lg")

            stats_md = gr.Markdown("*Click 'Preview Keyframe' to see a test frame and estimated stats.*")
            preview_img = gr.Image(label="Test Keyframe Preview", height=400)

            gr.Markdown("---")
            kf_gallery = gr.Gallery(label="All Keyframes", columns=3, height=250)
            video_out = gr.Video(label="Generated Video")
            gen_status = gr.Markdown("")

    # --- Event handlers ---
    def refresh_chars():
        new_chars = get_trained_characters()
        return gr.update(choices=new_chars, value=new_chars[0] if new_chars else "(none)")

    refresh_btn.click(fn=refresh_chars, outputs=[char_dd])

    preview_btn.click(
        fn=preview_keyframe,
        inputs=[char_dd, width_sl, height_sl, sdxl_steps_sl, sdxl_guide_sl, lora_sl,
                kf_tb, neg_tb, cogvid_steps_sl, cogvid_guide_sl, num_frames_sl, fps_sl, face_cb],
        outputs=[preview_img, stats_md],
    )

    generate_btn.click(
        fn=generate_video,
        inputs=[char_dd, width_sl, height_sl, sdxl_steps_sl, sdxl_guide_sl, lora_sl,
                kf_tb, motion_tb, neg_tb, motion_neg_tb,
                cogvid_steps_sl, cogvid_guide_sl, num_frames_sl, fps_sl, face_cb, denoise_sl],
        outputs=[kf_gallery, video_out, gen_status],
    )

demo.launch(share=True, debug=True)